# beam1d: the 1D base

Euler–Bernoulli beam of the hexagonal wrapper, the 1D counterpart of the Z3ST 3D cases. In the thesis it serves deliverable 1, the validity of the 1D beam against the 3D FEM (Z3ST as reference) for thermal bowing, and it is the base of Module 5 (comparison Timoshenko, eigenstrains and integral solution).

The theoretical set-up is the canonical one of NUBOW (McLennan, ANL-8068), the reference FORTRAN code for core bowing: a conventional beam loaded by an equivalent thermal moment, integrated as a cantilever (§1.6). beam1d follows it and adds the axial part, α(T) inside the section integral and Hermite elements. Pads (restrained bowing) are described qualitatively, as future work (§1.7, §2.3).

This notebook covers:
- **nomenclature** (§0): every symbol with its defining formula;
- **theory** (§1): the beam equations, the section built from `geometry.yaml`, the closed forms;
- **discretisation** (§2): Hermite beam and linear rod elements, Ma's finite differences, round-off;
- **practice** (§3): the section of a case and the layout of a case folder;
- **one section per case**: IAEA VP1 (§4: standard theory, discretisation, comparison with the IAEA document and with Z3ST 3D);
- **next steps** (§6).

```
sandbox/U_hex_beam/
  beam1d/            package: section, element, solver, thermal, case, fdm, mesh3d, __main__
  beam1d_VP1.ipynb   this notebook
  VP1/               geometry.yaml boundary_conditions.yaml input.yaml Allrun non-regression.py vp1_vs_z3st.py
```

Run it in the env `z3st` from this folder. The cells solve the cases through the package (`beam1d.__main__.run`). They also read what `VP1/Allrun` saved in `output/`: the checks and the comparison with Z3ST. Run it first.

References (in `documentazione/` unless marked): Öchsner, *Elasto-Plasticity of Frame Structure Elements* (2014), Ch. 4–5; Ma et al., NED 345 (2019) 28–39; IAEA IWGFR/75 (1990), Part I; McLennan, *NUBOW: A FORTRAN-IV Program for the Static Elastic Structural Analysis of Bowed Reactor Cores* (ANL-8068, 1974); Reddy, CMAME (1997) [Consensus]; ASN Guide No. 28.

## 0. Nomenclature

SI units throughout; a T difference in K equals one in °C. The formulas use only the symbols below. Code cells keep their Python names, listed where they differ.

**Axes and geometry**

| symbol | definition | unit | code |
|---|---|---|---|
| z | axial coordinate, from the clamp (z = 0) to the top (z = H) | m | `z` |
| x, y | section coordinates from the centroid; corners on ±y for `orientation: vertical` | m | `s.xy` |
| H | length of the wrapper | m | `c.H` |
| t | wall thickness | m | `s.t` |
| $D_o$ | outer corner-to-corner diameter, `D` of `geometry.yaml`: $D_o = 2a_o$ | m | `c.geo["D"]` |
| $a_o,\ a_i,\ a_m$ | outer, inner and mid-wall corner radius: $a_o = D_o/2$, $a_i = a_o - 2t/\sqrt3$, $a_m = (a_o + a_i)/2$ | m | `s.a_o`, `s.a_i`, `s.a_m` |
| D | width across which ΔT is taken (NUBOW's D); $D = D_m$ in VP1 | m | |
| $D_m$ | mid-wall corner-to-corner distance, $2a_m$ (0.150 m in VP1) | m | `D` in §4.1 and §4.2 |
| $F_o$ | outer across-flats width, $\sqrt3\,a_o$ | m | |
| $p_m$ | mid-wall apothem (centroid to flat), $\tfrac{\sqrt3}{2}a_m$ | m | |
| $\varphi_0$ | angle of corner 0; corner k is at $\varphi_0 + k\pi/3$ | rad | `s.phi0` |
| $s$ | arc length along the mid-wall ring | m | |
| $\mathbf X(s) = (x_c, y_c)$ | point of the mid-wall ring at s | m | |
| ξ | through the wall: normal distance from the mid-wall flat, outward, $-t/2 \le \xi \le t/2$; the wall point (s, ξ) is $(1 + \xi/p_m)\,\mathbf X(s)$, on the radial line through $\mathbf X(s)$ | m | |
| $z_b,\ z_t,\ L_c$ | core bottom, core top, core length $L_c = z_t - z_b$ | m | `zb`, `zt`, `L_c` |
| LRP, URP | lower and upper reference points (VP1: z = 3.0 and 4.0 m) | m | |

**Temperature and material**

| symbol | definition | unit | code |
|---|---|---|---|
| $T(x, y, z)$ | temperature | K | `T` |
| $T_\mathrm{ref}$ | stress-free temperature | K | `T_ref`, `Tref` |
| $T_\mathrm{in}$ | inlet temperature, the whole section below the core (VP1: 673.15 K) | K | `T_in` |
| $\Delta T$ | transverse T difference across the distance D (VP1: hot corner − cold corner) | K | `dT` |
| $\bar T$ | section mean, $\frac{1}{A_w}\int_A T\,dA$ | K | |
| $\Delta T_m$ | rise of $\bar T$ above the core, $\bar T - T_\mathrm{in}$ (VP1: 125 K) | K | `dT_m` |
| $\mathbf G$ | moment-weighted T gradient, $\frac{1}{I_w}\int_A T\,(x, y)\,dA$; g is one of its components | K/m | `g` |
| $T_c,\ T_0$ | T at the centroid | K | `T0` |
| $\eta(z)$ | core ramp, $\min(\max((z - z_b)/L_c,\,0),\,1)$ | – | `r`, `ramp` |
| E, ν | Young's modulus, Poisson's ratio | Pa, – | |
| λ, μ | Lamé constants (3D model only) | Pa | |
| $\alpha_t$ | tangent expansion coefficient, $d\varepsilon^*/dT$ | 1/K | |
| $\bar\alpha$ | secant coefficient from $T_\mathrm{ref}$: $\bar\alpha(T) = \frac{1}{T - T_\mathrm{ref}}\int_{T_\mathrm{ref}}^{T}\alpha_t\,dT'$ | 1/K | `alpha` |
| α | $\bar\alpha$ when constant (VP1: 1.8e-5) | 1/K | `alpha` |
| $\rho$ | density | kg/m³| |
|  $g_n$ | standard gravity (9.81 m/s²) | m/s² | |


**Section** (the undeformed wall Ω₀, the same at every z)

| symbol | definition | unit | code |
|---|---|---|---|
| $\int_A (\cdot)\,dA$ | integral over the wall section Ω₀ | | `s.dA @ …` |
| $A_w$ | wall area, $\int_A dA$ | m² | `s.A` |
| $S_x,\ S_y$ | $\int_A x\,dA$, $\int_A y\,dA$ (zero: the axis is the centroid) | m³ | `s.S_x`, `s.S_y` |
| $I_w$ | $\int_A y^2\,dA = \int_A x^2\,dA$ ($I_{xx} = I_{yy}$ for the regular hexagon) | m⁴ | `s.I` |
| $I_{xy}$ | $\int_A xy\,dA$ (zero) | m⁴ | `s.I_xy` |
| $r_g$ | radius of gyration, $\sqrt{I_w/A_w}$ | m | |
| EA, EI | $E A_w$, $E I_w$ (uniform E) | N, N m² | `EA`, `EI` |
| $B_0,\ \mathbf B_1,\ \mathbf B_2$ | $\int_A E\,dA$, $\int_A E\,(x, y)\,dA$, $\int_A E\,(x, y)(x, y)^T dA$ | N, N m, N m² | |

**Free strains and thermal resultants**

| symbol | definition | unit | code |
|---|---|---|---|
| $\varepsilon^*$ | thermal eigenstrain, $\bar\alpha(T)\,(T - T_\mathrm{ref})$ | – | `eps` |
| $\varepsilon_0^*$ | free axial strain, $\frac{1}{A_w}\int_A \varepsilon^*\,dA$ | – | `s.mean` |
| $\kappa^*_a$ | free curvature of plane a = x, y: $\frac{1}{I_w}\int_A \varepsilon^*\,a\,dA$ | 1/m | `s.curvature` |
| $\varepsilon_r$ | residual strain, $\varepsilon^* - \varepsilon_0^* - \kappa^*_x x - \kappa^*_y y$ | – | `eps_r` |
| $N_T$ | thermal force, $\int_A E\varepsilon^*\,dA = EA\,\varepsilon_0^*$ | N | |
| $M_{T,a}$ | thermal moment of plane a, $\int_A E\varepsilon^*\,a\,dA = EI\,\kappa^*_a$ | N m | |
| k | free curvature of the axis, $-\kappa^*_a$: $u_a'' = k$ in a free bow | 1/m | `k` |
| $k_0$ | plateau of k (VP1: $\alpha\Delta T/D_m$ = 6.0e-3) | 1/m | `k0` |
| $\hat M$ | thermal moment as the FE load, $EI\,k = -M_{T,a}$ | N m | |
| C | NUBOW weighting function, $C = \lVert\mathbf G\rVert\,D/\Delta T$ (1 for T linear across the section) | – | |
| c | thermal-shear coefficient, $\gamma = -c\,k' = +c\,\kappa^{*\prime}_a$ (hexagon: $0.410\,a_m^2$) | m² | `s.thermal_shear()` |

**Displacements, strains and internal forces**

| symbol | definition | unit | code |
|---|---|---|---|
| $(\cdot)'$ | $d/dz$; on $\alpha_t$ (§1.4, case b) it is $d/dT$ | | |
| $(\cdot)_{,i}$, $(\cdot)_{,ij}$ | comma notation: the indices after the comma are partial derivatives, 1, 2, 3 ≡ x, y, z; e.g. $\varepsilon^*_{,yz} = \partial^2\varepsilon^*/\partial y\,\partial z$, $\varepsilon^*_{11,23} = \partial^2\varepsilon^*_{11}/\partial x_2\,\partial x_3$ | | |
| $u_x,\ u_y,\ u_z$ | displacements of the centroid| m | `U_x`, `U_y`, `U_z` |
| $u_a$ | $u_x$ or $u_y$, bending plane a | m | |
| w | deflection in the bending plane in use: $w \equiv u_y$ in VP1, $w \equiv u_x$ in Ma 2019 | m | `w` |
| θ | slope, $u_a'$ | rad | `du` |
| ε | fibre strain, $u_z' - x\,u_x'' - y\,u_y''$ | – | |
| $\sigma_{zz}$ | fibre stress, $E\,(\varepsilon - \varepsilon^*)$ | Pa | `sigma` |
| N | axial force, $\int_A \sigma_{zz}\,dA = EA\,(u_z' - \varepsilon_0^*)$ | N | `Nm` |
| $m_a$ | bending moment, $-\int_A \sigma_{zz}\,a\,dA = EI\,(u_a'' + \kappa^*_a)$ | N m | |
| $q_z,\ q_a$ | axial and transverse load per unit length | N/m | |
| $R_1,\ R_2$ | $\int_0^z \eta\,d\zeta$, $\int_0^z R_1\,d\zeta$ | m, m² | `R1`, `R2` |
| γ | shear strain (3D only Timoshenko ) | – | |
| $\gamma^*$ | eigen shear of a Timoshenko beam, $-c\,k'$ (§1.4, §6) | – | |

**Discretisation**

| symbol | definition | unit | code |
|---|---|---|---|
| $L_e$ | element length; the grid step of a finite-difference scheme | m | `h`, `L` |
| $L_i$ | length of segment or element i (NUBOW Eqs. 4–7, §1.6) | m | |
| ζ | integration variable along z; local coordinate on a segment in §1.6 | m | |
| n | number of elements | – | `N`, `nz` |
| ẑ | local coordinate of an element, $\hat z = (z - z_1)/L_e \in [0, 1]$ | – | `xi` |
| $\mathbf N(\hat z)$ | shape functions (bold: N alone is the axial force) | – | `element.shape` |
| $\varphi_i$ | nodal rotation, $\theta(z_i)$ (i ≥ 1; $\varphi_0$ is the corner angle above) | rad | |
| $K_e,\ f_e$ | element stiffness and load | | |
| K, f, U | global stiffness, load and nodal unknowns | | `K`, `f`, `U` |
| Q | reactions, $KU - f$ at the constrained dofs | N, N m | `R` |
| $n_g$ | Gauss points per element | – | `n_gauss` |
| P, $M_c$ | concentrated force and moment at a node | N, N m | |

**Sign conventions.** $m_a = EI\,(u_a'' + \kappa^*_a)$. A free bow has $m_a = 0$, so $u_a'' = -\kappa^*_a = k$. The side with the larger $\varepsilon^*$ lengthens and becomes convex, so the axis bows towards the side with the smaller $\varepsilon^*$.

**Symbols of the sources.** The text uses only the symbols above; this table is for reading the sources.

| source | their symbol → ours |
|---|---|
| Ma et al. 2019 | x → z; z → x; w → $u_x$; u, $u_0$ → $u_z$; $M = \int z\sigma\,dA$ → $-m_x$; $M_T$ → $M_{T,x}$; $N_T$ → $N_T$; $T_f$ → $T_\mathrm{ref}$; h → $L_e$; L → H; G (gravity) → $q_z$ |
| NUBOW (ANL-8068) | x → z; y → −y; $v_T$ → $u_y$; $M_T$ → $\hat M = -M_{T,y}$; D ("diameter or across-flats width") → D; ΔT ("across-flats temperature difference") → ΔT; C → C |
| IWGFR/75, App. 5 | Z → z; Y → $u_y$; s → $z - z_b$; λ → $L_c$; 1/ρ → k; k → $k_0$; 2a → $D_m$ |

In [ ]:
import json
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import yaml
from IPython.display import Image, Markdown, display

BASE = os.path.abspath(".")
sys.path.insert(0, BASE)
from beam1d import case as case_mod
from beam1d import element, fdm, solver, thermal
from beam1d.__main__ import run
from beam1d.mesh3d import section_from_msh
from beam1d.section import HexSection

VP1 = os.path.join(BASE, "VP1")
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.4})


def table(rows, head):
    # a list of rows as a markdown table
    lines = ["| " + " | ".join(head) + " |", "|" + "---|" * len(head)]
    lines += ["| " + " | ".join(str(v) for v in r) + " |" for r in rows]
    display(Markdown("\n".join(lines)))

## 1. Theory

### 1.1 Kinematics and Hooke's law with an eigenstrain

The axis is the centroid line, z from the clamp (z = 0) to the top (z = H); x and y are the section coordinates. The unknowns are the centroid displacements $u_x(z)$, $u_y(z)$ and $u_z(z)$. Bernoulli: plane sections stay plane and normal to the axis, so the fibre at (x, y) has

$$\varepsilon(x,y,z) = u_z' - x\,u_x'' - y\,u_y'' .$$

Hooke's law with the thermal eigenstrain:

$$\sigma_{zz} = E\,(\varepsilon - \varepsilon^*), \qquad \varepsilon^* = \bar\alpha(T)\,(T - T_\mathrm{ref}), \qquad \bar\alpha(T) = \frac{1}{T - T_\mathrm{ref}}\int_{T_\mathrm{ref}}^{T}\alpha_t\,dT' ,$$

with $\bar\alpha$ the secant expansion coefficient from $T_\mathrm{ref}$ and $\alpha_t$ the tangent one (both 1/K).

### 1.2 Equilibrium

Virtual work gives the internal virtual work $\delta W_\mathrm{int} = \int \left(N\,\delta u_z' + m_x\,\delta u_x'' + m_y\,\delta u_y''\right) dz$, with

$$N = \int_A \sigma_{zz}\,dA, \qquad m_a = -\int_A \sigma_{zz}\, a\, dA \quad (a = x, y).$$

Integrating by parts against the work of the loads per unit length $q_z$, $q_a$ (N/m), the equilibrium equations are

$$N' + q_z = 0, \qquad m_a'' = q_a .$$

### 1.3 Section resultants and free strains

Inserting $\sigma_{zz}$, with $B_0 = \int_A E\,dA$, $\mathbf B_1 = \int_A E\,(x, y)\,dA$, $\mathbf B_2 = \int_A E\,(x,y)(x,y)^T dA$ and $\mathbf u'' = (u_x'', u_y'')$:

$$N = B_0 u_z' - \mathbf B_1\cdot \mathbf u'' - N_T, \qquad N_T = \int_A E\varepsilon^* dA, \qquad \mathbf M_T = (M_{T,x}, M_{T,y}) = \int_A E\varepsilon^*\,(x, y)\,dA .$$

With E uniform over the section, $\mathbf B_1 = E\,(S_x, S_y) = 0$ (the axis is the centroid), $\mathbf B_2 = EI_w\mathbb 1$, and the three problems decouple:

$$N = EA\,(u_z' - \varepsilon_0^*), \qquad m_a = EI\,(u_a'' + \kappa^*_a),$$

$$\varepsilon_0^* = \frac{1}{A_w}\int_A \varepsilon^*\,dA, \qquad \kappa^*_a = \frac{1}{I_w}\int_A \varepsilon^*\,a\,dA \quad (a = x, y).$$

### 1.4 Transverse thermal gradient in a clamped beam

A temperature field $T(x, y, z)$ (K) over the section is turned into the displacements of the axis in the steps below. All integrals are over the undeformed section Ω₀ (§1.5).

**1. Free thermal strain of each fibre.**

$$\varepsilon^*(x,y,z) = \int_{T_\mathrm{ref}}^{T} \alpha_t(T'; x, y)\,dT' = \bar\alpha(T)\,(T - T_\mathrm{ref}) \qquad (-),$$

so that $\alpha_t = d\varepsilon^*/dT = \bar\alpha + (T - T_\mathrm{ref})\,d\bar\alpha/dT$.

**2. Projection on plane sections.** The axis carries only strains linear in (x, y) (§1.1):

$$\varepsilon^* = \varepsilon_0^* + \kappa^*_x\,x + \kappa^*_y\,y + \varepsilon_r, \qquad \int_A \varepsilon_r\,dA = \int_A \varepsilon_r\,x\,dA = \int_A \varepsilon_r\,y\,dA = 0 .$$

With $S_x = S_y = 0$, $I_{xy} = 0$ and $I_{xx} = I_{yy} = I_w$ (§1.5), the three conditions give the integrals of §1.3: $\varepsilon_0^*$ (−) and $\kappa^*_a$ (1/m). $\varepsilon_r$ is the part of the strain the axis cannot follow.

**3. Equivalent thermal force and moment.** With E uniform,

$$N_T = \int_A E\,\varepsilon^*\,dA = EA\,\varepsilon_0^* \;\;(\mathrm N), \qquad M_{T,a} = \int_A E\,\varepsilon^*\,a\,dA = EI\,\kappa^*_a \;\;(\mathrm{N\,m}).$$

They are the loads of the eigenstrain on the axis (the thermal terms of $f_e$, §2.1). $\varepsilon_r$ has no resultant.

**4. Clamped–free beam.** Clamp at z = 0 ($u_x = u_y = u_z = 0$, $u_x' = u_y' = 0$), free top, no load. The beam is statically determinate: N ≡ 0, m ≡ 0, and §1.3 gives $u_z' = \varepsilon_0^*$, $u_a'' = -\kappa^*_a = k$ (a = x, y). Integrated from the clamp:

| quantity | expression | unit |
|---|---|---|
| axial displacement | $u_z(z) = \int_0^z \varepsilon_0^*\,d\zeta$ | m |
| curvature of the axis | $u_a''(z) = -\kappa^*_a(z)$ | 1/m |
| slope | $\theta(z) = u_a'(z) = -\int_0^z \kappa^*_a\,d\zeta$ | rad |
| deflection | $u_a(z) = -\int_0^z (z - \zeta)\,\kappa^*_a(\zeta)\,d\zeta$ | m |
| fibre stress | $\sigma_{zz} = -E\,\varepsilon_r$ | Pa |

The deflection is the double integral in Cauchy's form. EI drops out of u: it sets only the internal forces and the reactions, here zero. The side where $\varepsilon^*$ is larger lengthens and becomes convex: the axis bows towards the side where $\varepsilon^*$ is smaller, along $-\boldsymbol\kappa^*$.

**5. Fully restrained beam.** Both ends clamped and axially fixed, with $\varepsilon_0^*$ and $\kappa^*_a$ uniform along z: u ≡ 0 satisfies equilibrium and all the supports, so $N = -N_T$ and $m_a = M_{T,a}$. These are the force and moment the supports supply.

**Case (a): α constant.** $\varepsilon^* = \alpha\,(T - T_\mathrm{ref})$ is linear in T, so the projection acts on T itself:

$$\varepsilon_0^* = \alpha\,(\bar T - T_\mathrm{ref}), \qquad \boldsymbol\kappa^* = \alpha\,\mathbf G, \qquad \bar T = \frac{1}{A_w}\int_A T\,dA, \quad \mathbf G = \frac{1}{I_w}\int_A T\,(x, y)\,dA \;\;(\mathrm{K/m}),$$

with $\mathbf G$ the moment-weighted gradient of T over the section. $T_\mathrm{ref}$ enters $\boldsymbol\kappa^*$ as $-\alpha\,T_\mathrm{ref}\int_A (x, y)\,dA/I_w$, which is zero because $S_x = S_y = 0$ (the axis is the centroid). $T_\mathrm{ref}$ does remain in $\varepsilon_0^*$.
- **T linear in (x, y):** take $T = T_0 + \mathbf g\cdot(x, y)$. $S_x = S_y = 0$ gives $\bar T = T_0$. $I_{xy} = 0$ and $I_{xx} = I_{yy} = I_w$ give $\mathbf G = \mathbf g$. Then $\varepsilon^* = \varepsilon_0^* + \boldsymbol\kappa^*\cdot(x, y)$ lies in span{1, x, y}, so $\varepsilon_r = 0$, and $\sigma_{zz} = -E\,\varepsilon_r \equiv 0$ in the clamped–free beam (step 4, N ≡ 0 and m ≡ 0). In the fully restrained beam (step 5), $\sigma_{zz} \neq 0$. The isotherms are lines normal to $\mathbf G$. Two of them a distance D apart differ by $\Delta T = \lVert\mathbf G\rVert D$, so $\lVert\boldsymbol\kappa^*\rVert = \alpha\,\Delta T/D$ for any section shape (for a section without $I_{xy} = 0$ and $I_{xx} = I_{yy}$, the projection uses the full inertia tensor). This is NUBOW's equivalent thermal moment with weighting function C = 1 (McLennan, ANL-8068, p. 12, Eq. 2), in our notation $\lVert\mathbf M_T\rVert = EI\,C\,\alpha\,\Delta T/D$ (on VP1, §4.1 checks $-\kappa^*_y D_m/(\alpha\Delta T) = 1$).
- **T not linear:** $\varepsilon_r = \alpha\,[T - \bar T - \mathbf G\cdot(x, y)] \neq 0$ and $\sigma_{zz} = -E\,\varepsilon_r$. NUBOW's weighting function is then $C = (\mathbf G\cdot\mathbf n)\,D/\Delta T$, with ΔT the T difference over a chord of length D in the direction $\mathbf n$ (in VP1, corner to corner, $D = D_m$). C is the ratio of the moment-weighted gradient to the chord gradient.

**Compatibility in 3D.** A strain field ε is compatible when a continuous, single-valued displacement $\mathbf u$ exists with $\varepsilon = \mathrm{sym}\,\nabla\mathbf u$.
- A compatible eigenstrain is taken by a free body without stress: $\varepsilon = \varepsilon^*$, σ = 0 satisfies equilibrium and the traction-free boundary.
- An incompatible eigenstrain cannot be followed by a continuous body. The free fibres would leave gaps or overlap, continuity forces $\varepsilon \neq \varepsilon^*$, and $\varepsilon - \varepsilon^*$ is an elastic strain. The result is a self-equilibrated stress, with no load and no support.

In a simply connected body, ε* is compatible if and only if it satisfies the six Saint-Venant equations

$$\frac{\partial^2 \varepsilon^*_{ij}}{\partial x_k\,\partial x_l} + \frac{\partial^2 \varepsilon^*_{kl}}{\partial x_i\,\partial x_j} - \frac{\partial^2 \varepsilon^*_{ik}}{\partial x_j\,\partial x_l} - \frac{\partial^2 \varepsilon^*_{jl}}{\partial x_i\,\partial x_k} = 0 .$$

For an isotropic eigenstrain $\varepsilon^*_{ij} = \varepsilon^*\,\delta_{ij}$, with $\varepsilon^*_{,ij} = \partial^2\varepsilon^*/\partial x_i\,\partial x_j$ (comma notation, §0; $x_1, x_2, x_3 = x, y, z$):
- (i, j, k, l) = (1, 1, 2, 2) gives $\varepsilon^*_{,11} + \varepsilon^*_{,22} = 0$, and likewise for the pairs (1, 3) and (2, 3). Together the three equations give $\varepsilon^*_{,11} = \varepsilon^*_{,22} = \varepsilon^*_{,33} = 0$.
- (i, j, k, l) = (1, 1, 2, 3) gives $\varepsilon^*_{11,23} = 0$ (the off-diagonal terms vanish), that is $\varepsilon^*_{,23} = 0$, and likewise $\varepsilon^*_{,13} = \varepsilon^*_{,12} = 0$.

All second derivatives vanish: an isotropic ε* is compatible if and only if it is linear in (x, y, z). With α constant, this means T linear in (x, y, z).

Applied to the VP1 field, the (1, 1, 2, 3) equation fails in the core and the ramp ends carry δ layers (§4.1).

The wall is a tube, a multiply connected body, so the six equations are necessary but not sufficient. They give $\mathbf u$ only locally. Around a loop enclosing the hole, $\mathbf u$ can fail to close (a Volterra dislocation), and the Cesàro line integrals around the hole must also vanish. For $\varepsilon^* = a + \mathbf b\cdot\mathbf x$, the displacement $u_i = a\,x_i + (\mathbf b\cdot\mathbf x)\,x_i - \tfrac12\,b_i\,\lvert\mathbf x\rvert^2$ has $\mathrm{sym}\,\nabla\mathbf u = \varepsilon^*\,\mathbf 1$. It is a polynomial, hence single-valued, so the Cesàro conditions hold.

The beam result $\varepsilon_r = 0$ concerns only $\sigma_{zz}$ under plane sections. An incompatible ε* also leaves in-plane and shear stresses, which the beam does not contain.

**Case (b): α(T), α(x, y).** $\varepsilon^*$ is no longer linear in T, so a T linear across the section also leaves a residual. Take $T = T_c + g\,x$ ($T_c$ the centroid T, g in K/m), with $\alpha_t$ smooth over the T range of the section and $\alpha_t$ a function of T only. Expanding about $T_c$ ($\alpha_t$ and its derivatives at $T_c$, with $d\varepsilon^*/dT = \alpha_t$ from step 1):

$$\varepsilon^* = \varepsilon^*(T_c) + \alpha_t\,g\,x + \tfrac12\,\alpha_t'\,g^2x^2 + \tfrac16\,\alpha_t''\,g^3x^3 + \dots$$

The x and y axes are mirror lines of the regular hexagon, in both orientations. Hence:
- $\int_A x\,dA = 0$, because the axis is the centroid;
- $\int_A x^3 dA = 0$, because the integrand is odd in x;
- $\int_A x^n y\,dA = 0$, because the integrand is odd in y;
- $\int_A x^2 dA = I_w$ and $\int_A x^4 dA > 0$.

Projecting term by term (§1.4 step 2):

$$\kappa^*_x = \alpha_t\,g + \frac{\alpha_t''\,g^3}{6I_w}\int_A x^4\,dA, \qquad \kappa^*_y = 0, \qquad \varepsilon_0^* = \varepsilon^*(T_c) + \tfrac12\,\alpha_t'\,g^2\,\frac{I_w}{A_w}, \qquad \varepsilon_r \simeq \tfrac12\,\alpha_t'\,g^2\left(x^2 - \frac{I_w}{A_w}\right).$$

- $\kappa^*_y = 0$: every term times y is of the form $x^n y$.
- $\kappa^*_x$: only the odd terms survive. The $x^2$ term times x gives $x^3$, whose integral is zero.
- $\varepsilon_0^*$: only the even terms survive. The $g^4$ term is dropped. A linear T with $\alpha_t' > 0$ lengthens the axis by more than $\varepsilon^*(T_c)$.
- $\varepsilon_r$ is ε* minus its projection, to order $g^2$. $x^2$ is not in span{1, x, y}, so $\varepsilon_r \neq 0$ unless $\alpha_t' = 0$, that is, unless $\varepsilon^*(T)$ is a straight line. Subtracting $I_w/A_w$ makes the mean zero: $\int_A (x^2 - I_w/A_w)\,dA = 0$. $\int_A \varepsilon_r\,x\,dA = \int_A \varepsilon_r\,y\,dA = 0$ by symmetry.
- A linear T then gives a stress $\sigma_{zz} \simeq -\tfrac12 E\,\alpha_t'\,g^2\,(x^2 - I_w/A_w)$ through the curvature of $\varepsilon^*(T)$. In 3D, $\varepsilon^*_{,xx} = \partial^2\varepsilon^*/\partial x^2 = \alpha_t'(T)\,g^2 \neq 0$, so this field is incompatible as well.
- To first order in g, $\kappa^*_x$ follows the **tangent** coefficient at the centroid temperature, $\alpha_t(T_c)\,g$. The secant estimate $\bar\alpha(T_c)\,g$ differs from it by $(T_c - T_\mathrm{ref})\,g\,d\bar\alpha/dT|_{T_c}$, which is zero only where $d\bar\alpha/dT = 0$ at $T_c$ or for $T_c = T_\mathrm{ref}$. $T_\mathrm{ref}$ drops out of $\boldsymbol\kappa^*$ exactly: changing it shifts ε* by a constant.
- α varying with (x, y) at the same T (two materials with the same E): $\varepsilon^*$ is evaluated fibre by fibre and steps 2–5 are unchanged. $\mathbf G$ no longer gives $\boldsymbol\kappa^*$, and $T_\mathrm{ref}$ no longer drops out of it: a uniform $T \neq T_\mathrm{ref}$ bends the beam if $\int_A \alpha\,(x, y)\,dA \neq 0$.

The expansion explains the result; the code does not use it. `HexSection.mean` and `HexSection.curvature` compute $\varepsilon^*(T)$ at each quadrature point of the section and integrate it numerically, with no Taylor series and no assumption on the smoothness of α. The expansion requires $\alpha_t$ to be differentiable over the T range of the section. A tabulated $\bar\alpha$, linear between table points, has $d\bar\alpha/dT$ jumping at each table point, so $\alpha_t = \bar\alpha + (T - T_\mathrm{ref})\,d\bar\alpha/dT$ jumps there too. The expansion then holds only if the whole section lies within one table interval; the numerical integral holds in every case.

An E that depends on T or on (x, y) in general gives $\mathbf B_1 \neq 0$ (§1.3): the modulus-weighted centroid moves and N couples with $m_a$. It is not covered here.

**What the beam does not see.** The 3D model adds the in-plane stresses and, where $\boldsymbol\kappa^*$ varies along z, the thermal shear $\gamma = -c\,k'$ (`HexSection.thermal_shear`, $c = 0.410\,a_m^2$ for the regular hexagon).

#### Figure 1.4: what the axis follows and what is left

Context. The beam keeps only two numbers per section, $\varepsilon_0^*$ (axial strain) and $\boldsymbol\kappa^*$ (curvature), the projection of $\varepsilon^*$ on span{1, x, y} (§1.4, step 2). The figure shows this projection on the real VP1 section and what it leaves out, $\varepsilon_r$, for the cases (a) and (b) above.

The three T fields are invented; they are not VP1 data. Parameters: α = 1.8e-5 1/K, $\alpha_t'$ = 7e-9 1/K², $T_c - T_\mathrm{ref}$ = 100 K, g = 400 K/m. One column per case:
- top: $\varepsilon^*$ of each fibre (dots) and the plane section $\varepsilon_0^* + \kappa^*_x x$ (line), the only part the axis follows;
- bottom: the rest, $\varepsilon_r$. It has no force and no moment, so it does not bend the axis, but it gives $\sigma_{zz} = -E\,\varepsilon_r$.

Reading: $\varepsilon_r = 0$ only in the first column (α constant, T linear). A non-linear T (second) leaves a large $\varepsilon_r$; α(T) with a linear T (third) leaves a small parabolic one, $\tfrac12\alpha_t' g^2 (x^2 - I_w/A_w)$.

The y axes of the bottom row are not shared: $\varepsilon_r$ is about 600 times smaller in the third column than in the second.
- Second column: T itself departs from a line by tens of K, so $\varepsilon_r \approx \alpha\,\delta T \approx 1.8\cdot10^{-5} \times 40 \approx 7\cdot10^{-4}$ (first order).
- Third column: T is linear and only $\varepsilon^*(T)$ is curved. Across the section T spans $g\,x_\mathrm{max} \approx 26$ K, over which $\alpha_t$ changes by $\alpha_t' \times 26 \approx 1.8\cdot10^{-7}$ 1/K, about 1 % of α. Then $\tfrac12\alpha_t' g^2 = 5.6\cdot10^{-4}$ 1/m² times $\lvert x^2 - I_w/A_w\rvert \lesssim 2.3\cdot10^{-3}$ m² ($I_w/A_w = r_g^2$, $r_g \approx 48$ mm) gives $\varepsilon_r \sim 10^{-6}$ (second order, $\propto g^2$).


In [ ]:
# §1.4 steps 2 and 4 on the VP1 section, invented T fields (not VP1 data): eps* of every fibre (dots)
# and what the axis can follow (line), bending along x
s = case_mod.load_case(VP1).section
x, y = s.xy[:, 0], s.xy[:, 1]
a0, da, dT0, g = 1.8e-5, 7e-9, 100.0, 400.0   # invented: alpha (1/K), alpha_t' (1/K^2), T_c - T_ref (K), g (K/m)
x_f = s.a_m * np.cos(np.pi / 6)                             # mid-wall flats at x = -/+ x_f (m)
bump = lambda p: dT0 * np.exp(-((p[0] + x_f) / 0.02) ** 2)  # hot flat at -x (K above T_ref)
cases = [
    ("(a) α constant, T = T_c + g x (linear)", lambda p: a0 * (dT0 + g * p[0]) + 0 * p[1]),
    ("(a) α constant, T not linear: hot flat at −x", lambda p: a0 * bump(p) + 0 * p[1]),
    ("(b) α(T) with α_t linear in T, T = T_c + g x", lambda p: a0 * (dT0 + g * p[0]) + 0.5 * da * (dT0 + g * p[0]) ** 2 + 0 * p[1]),
]
xx = np.linspace(x.min(), x.max(), 50)
fig, ax = plt.subplots(2, 3, figsize=(15, 7), sharex=True)
for j, (name, eps_j) in enumerate(cases):
    e = s.field(eps_j, [0.0])[:, 0]
    e0, (kx, ky) = s.mean(eps_j, [0.0])[0], s.curvature(eps_j, [0.0])[0]
    e_r = e - e0 - kx * x - ky * y
    ax[0, j].scatter(x * 1e3, e * 1e3, s=3, c="0.5", label=r"$\varepsilon^*$ of the fibres")
    ax[0, j].plot(xx * 1e3, (e0 + kx * xx) * 1e3, "r", lw=2, label=r"$\varepsilon_0^* + \kappa^*_x x$ (plane section)")
    ax[0, j].axhline(e0 * 1e3, color="r", ls=":", lw=1)
    ax[0, j].set_title(name, fontsize=10)
    ax[1, j].scatter(x * 1e3, e_r * 1e6, s=3, c="tab:blue", label=r"$\varepsilon_r$ (quadrature)")
    if j == 2:
        ax[1, j].plot(xx * 1e3, 0.5 * da * g**2 * (xx**2 - s.I / s.A) * 1e6, "k--", lw=1.5,
                      label=r"$\frac{1}{2}\alpha_t' g^2 (x^2 - I_w/A_w)$")
        print(f"(b): max |eps_r - 1/2 alpha_t' g^2 (x^2 - I_w/A_w)| = "
              f"{np.abs(e_r - 0.5 * da * g**2 * (x**2 - s.I / s.A)).max():.1e} (exact: alpha_t'' = 0)")
    if j == 1:
        C = abs(kx) * 2 * x_f / (a0 * (bump((-x_f,)) - bump((x_f,))))
        ax[0, j].text(0.97, 0.95, f"C = |κ*ₓ| D / (α ΔT) = {C:.3f}\nD = 2 x_f, ΔT flat to flat",
                      transform=ax[0, j].transAxes, va="top", ha="right", fontsize=9, bbox=dict(fc="w", ec="0.7", alpha=0.9))
    if j == 0:
        ax[1, j].set_ylim(-1, 1)   # eps_r is round-off (~1e-18): plotted on the same 1e-6 scale it lies on zero
    verdict = ["ε_r = 0 (max {:.0e}, round-off)\n→ no stress",
               "max |ε_r| = {:.1e}\n→ σ_zz = −E ε_r ≠ 0",
               "max |ε_r| = {:.1e}, small but ≠ 0\n→ σ_zz = −E ε_r ≠ 0"][j]
    ax[1, j].text(0.5, 0.97, verdict.format(np.abs(e_r).max()), transform=ax[1, j].transAxes, va="top", ha="center",
                  fontsize=9, bbox=dict(fc="w", ec="0.7", alpha=0.9))
    ax[1, j].set_xlabel("x (mm)")
ax[0, 0].set_ylabel(r"$\varepsilon^*$ ($10^{-3}$)"); ax[1, 0].set_ylabel(r"$\varepsilon_r$ ($10^{-6}$)")
ax[0, 0].legend(fontsize=8, loc="lower right"); ax[1, 2].legend(fontsize=8, loc="lower center")
fig.suptitle("Invented T fields (not VP1 data) on the VP1 section, clamped–free beam\n"
             "Top: $\\varepsilon^*$ of each fibre and the plane section the axis follows (dotted: $\\varepsilon_0^*$). "
             "Bottom: the rest, $\\varepsilon_r$: it does not bend the axis but gives $\\sigma_{zz}$", fontsize=11)
plt.tight_layout()

### 1.5 The section: the integrated quantities

The beam does not see the shape of the section, only a few integrals over the undeformed wall Ω₀. With small strains Ω₀ is the same at every z and is not updated. The integrals fall into two groups.

**Geometric integrals** (no T, computed once):

| quantity | integral over Ω₀ | where it enters | regular hexagon |
|---|---|---|---|
| $A_w$ | $\int dA$ | axial stiffness EA; normalises $\varepsilon_0^*$ | $\frac{3\sqrt3}{2}(a_o^2 - a_i^2) = 6 a_m t$ |
| $S_x,\ S_y$ | $\int x\,dA$, $\int y\,dA$ | position of the axis: zero when x, y are measured from the centroid | 0 |
| $I_{xx},\ I_{yy}$ | $\int y^2 dA$, $\int x^2 dA$ | bending stiffness EI; normalises $\kappa^*_a$ | $I_w = \frac{5\sqrt3}{16}(a_o^4 - a_i^4)$, any axis (6-fold symmetry) |
| $I_{xy}$ | $\int xy\,dA$ | couples the two bending planes | 0 |
| $B_0$ | $\int E\,dA$ | N from $u_z'$ (§1.3) | $EA_w$ |
| $\mathbf B_1$ | $\int E\,(x, y)\,dA$ | couples N with $\mathbf u''$ and $m_a$ with $u_z'$ | $E\,(S_x, S_y) = 0$ |
| $\mathbf B_2$ | $\int E\,(x, y)(x, y)^T dA$ | $m_a$ from $\mathbf u''$ | $EI_w\mathbb 1$ |

The B's are A, S and I weighted by the modulus: for E uniform over the section they are E times them. The zeros of the hexagon split the beam into one axial and two independent bending problems (§1.3). An E that varies over the section in general gives $\mathbf B_1 \neq 0$ and couples them; beam1d does not implement it.

**Thermal integrals** (of $\varepsilon^*(x, y, z)$, at every z):

| quantity | integral over Ω₀ | meaning |
|---|---|---|
| $\varepsilon_0^*$ | $\frac{1}{A_w}\int \varepsilon^*\,dA$ | free axial strain: lengthening of the axis |
| $\kappa^*_a$ | $\frac{1}{I_w}\int \varepsilon^*\,a\,dA$ | free curvature: bowing of the axis |
| $N_T,\ M_{T,a}$ | $\int E\varepsilon^*\,dA$, $\int E\varepsilon^*\,a\,dA$ | the same as a force and a moment: $EA\,\varepsilon_0^*$, $EI\,\kappa^*_a$ |

$\varepsilon_0^*$ and $\kappa^*_a$ are $A_w$ and $I_w$ with the weight $\varepsilon^*$. They are all the axis receives from the T field; the rest, $\varepsilon_r$, gives stress but no displacement (§1.4).

Code: `s.A`, `s.S_x`, `s.S_y`, `s.I`, `s.I_xy`, `s.modulus_weighted(E)`, `s.mean`, `s.curvature`.

**The wall Ω₀ from `geometry.yaml`.** The same numbers z3st meshes the 3D wall from:
- `D` is the outer corner-to-corner diameter $D_o$, so $a_o = D_o/2$;
- $a_i = a_o - 2t/\sqrt3$ (inner apothem = outer apothem − t);
- $a_m = a_o - t/\sqrt3$ (mid-wall);
- corners sharp, corner k at $\varphi_0 + k\pi/3$ ($\varphi_0 = 90°$ for `vertical`).

**How the integrals can be computed.**

| method | input | exact for | in beam1d |
|---|---|---|---|
| closed form | $a_o$, $a_i$ | the geometric integrals; the thermal ones for α constant and T linear ($\boldsymbol\kappa^* = \alpha\,\mathbf g$, §1.4) | checks |
| thin wall | mid-wall contour, $dA = t\,ds$ | none: $I_w \approx \tfrac52 a_m^3 t$, relative error $\approx -t^2/(3a_m^2)$ | not used |
| quadrature over Ω₀ | $\varepsilon^*$ at the quadrature points | polynomial integrands (below) | every integral |
| facets of the 3D mesh | the z3st `.msh` | the geometric integrals of the meshed section | check that the beam section is the 3D one (`mesh3d.section_from_msh`) |

**The quadrature over Ω₀.**
- *Pieces.* Ω₀ is six trapezoids, one per flat: inner and outer side parallel, the other two radial. Each flat is cut again at the fractions f = 1/4, 1/2, 3/4 along it (`HexSection.SPLITS`): 24 trapezoids.
- *Map.* Each trapezoid is mapped bilinearly onto the square [−1, 1]², r along the flat and q through the wall (`xi`, `eta` in `section.py`), as a Q4 finite element: $\int f\,dA = \iint f\,\lvert J\rvert\,dr\,dq$, with $\lvert J\rvert$ linear in q only.
- *Gauss–Legendre.* On [−1, 1], $\int f\,dr \approx \sum_{i=1}^{n} w_i\,f(r_i)$, with $r_i$ the roots of the Legendre polynomial $P_n$ and $w_i$ their weights. n points integrate exactly the polynomials of degree ≤ 2n − 1. Here n = 4 per direction (degree 7), 16 points per trapezoid, 384 in all.
- *Exactness.* x and y are bilinear in (r, q), so $\lvert J\rvert$, $x\lvert J\rvert$ and $x^2\lvert J\rvert$ have degree ≤ 3 per direction, and an $\varepsilon^*$ polynomial of low degree in (x, y) stays at most 7: these integrals are exact. A T with a kink or a step inside a trapezoid is not a polynomial there. The cuts are placed where the contour tables of `beam1d.thermal` can have one (corners, quarter points, mid-flat). The section integrates $\varepsilon^*$, not T: with `ma12` (T constant on each trapezoid) it is exact for any α(T); with `contour_linear` it is exact for α constant, while a tabulated α(T) adds kinks wherever T crosses a table point, which no cut follows. Those cases, and any other T, are integrated to quadrature accuracy.
- *Radial cuts.* The inner and outer hexagons are concentric and similar, so the points at fraction f on the two faces have the same polar angle and the line r = const is radial. Every quadrature point has one flat k and one fraction f, the coordinates of a T given on the mid-wall contour and constant through the wall.
- These are not the Gauss points along z ($n_g$ per element, §2), which integrate the element stiffness and load.

**T uniform through the wall.** A wall point has the coordinates (s, ξ) of §0: s along the mid-wall ring, ξ the normal distance from the mid-wall flat. It lies on the radial line through the ring point $\mathbf X(s) = (x_c, y_c)$:

$$(x, y) = \left(1 + \frac{\xi}{p_m}\right)(x_c, y_c), \qquad dA = \left(1 + \frac{\xi}{p_m}\right)d\xi\,ds .$$

The factor $1 + \xi/p_m$ is the scale of the hexagon through the point, whose flat is at $p_m + \xi$ from the centroid. Take α constant and T linear along the ring and constant through the wall, $T = T_0 + g\,x_c(s)$. The $T_0$ term has no moment ($S_x = 0$), and

$$\int_A T\,x\,dA = g\oint x_c^2\,ds\int_{-t/2}^{t/2}\left(1 + \frac{\xi}{p_m}\right)^2 d\xi, \qquad I_w = \oint x_c^2\,ds\int_{-t/2}^{t/2}\left(1 + \frac{\xi}{p_m}\right)^3 d\xi .$$

The ring integral cancels in $\kappa^*_x = \alpha\int_A T\,x\,dA / I_w$; with $e = t/(2p_m)$,

$$\kappa^*_x = \alpha g\,\frac{1 + e^2/3}{1 + e^2} \approx \alpha g\left(1 - \tfrac23 e^2\right),$$

slightly below $\alpha g$. Along a radial line x grows outward while T keeps its mid-wall value, so the outer face is colder and the inner face hotter than with $T = T_0 + g\,x$.

### 1.6 Closed forms

**Ramp of curvature.** A cantilever whose axis curvature $k = -\kappa^*_a$ is zero below $z_b$, rises linearly to $k_0$ at $z_t$ ($L_c = z_t - z_b$) and stays constant above. §1.4 step 4 gives

$$w = \begin{cases} 0 & z \le z_b \\ k_0 (z - z_b)^3 / (6L_c) & z_b < z \le z_t \\ k_0\left[L_c^2/6 + L_c (z - z_t)/2 + (z - z_t)^2/2\right] & z > z_t \end{cases}$$

and, for $\varepsilon_0^*$ ramping the same way to $\varepsilon_{0,\max}^*$, $u_z = \varepsilon_{0,\max}^*\int_0^z \eta\,d\zeta$.

**Element level.** A cantilever with a tip force has $w(H) = PH^3/3EI$ and $w'(H) = PH^2/2EI$. A constant axis curvature k gives $w = k z^2/2$, which equals two opposite nodal moments ±EIk on each element.

**What EI does.** In free bowing w depends on κ* only. With restraints (pads, §1.7) EI would also set the contact forces.

**Segment by segment: NUBOW Eqs. 4–7.** NUBOW (ANL-8068, pp. 12–13) splits the beam into segments of constant section, with the thermal moment linear on each one (Fig. 5), and integrates them as cantilevers one after the other. The result is the same closed form, built segment by segment.

*1. The trapezoid as two triangles.* On a segment of length $L_e$, with local coordinate $\zeta \in [0, L_e]$ from its lower end, the moment is

$$\hat M(\zeta) = \hat M_0 + (\hat M_1 - \hat M_0)\,\frac{\zeta}{L_e} = \hat M_0\left(1 - \frac{\zeta}{L_e}\right) + \hat M_1\,\frac{\zeta}{L_e} .$$

The first triangle is $\hat M_0$ at the lower end and 0 at the upper one; the second is 0 at the lower end and $\hat M_1$ at the upper one. There is no external load: the segment bends freely with curvature $w'' = \hat M/EI$. The M/EI diagram is a trapezoid, the sum of the two triangles. The problem is linear, so their contributions add.

*2. Slope at the upper end (first moment-area theorem).* The rotation is the area of the M/EI diagram:

$$\theta_1 = \frac{1}{EI}\int_0^{L_e}\hat M\,d\zeta = \frac{1}{EI}\left(\frac{\hat M_0 L_e}{2} + \frac{\hat M_1 L_e}{2}\right) = \frac{(\hat M_0 + \hat M_1)\,L_e}{2EI} \qquad \text{(NUBOW Eq. 5)}.$$

Each triangle has area $\hat M L_e/2$, so both ends weigh the same in θ.

*3. Deflection at the upper end (second moment-area theorem).* The deflection is the first moment of the M/EI diagram about the upper end:

$$w_1 = \frac{1}{EI}\int_0^{L_e}(L_e - \zeta)\,\hat M(\zeta)\,d\zeta .$$

Here the two triangles weigh differently, because their centroids have different lever arms to the upper end:

| triangle | area | centroid (from ζ = 0) | lever arm to the upper end | contribution × EI |
|---|---|---|---|---|
| peak $\hat M_0$ at the lower end | $\hat M_0 L_e/2$ | $L_e/3$ | $2L_e/3$ | $\hat M_0 L_e^2/3$ |
| peak $\hat M_1$ at the upper end | $\hat M_1 L_e/2$ | $2L_e/3$ | $L_e/3$ | $\hat M_1 L_e^2/6$ |

In full, $\int_0^{L_e}(L_e - \zeta)(1 - \zeta/L_e)\,d\zeta = L_e^2/3$ and $\int_0^{L_e}(L_e - \zeta)(\zeta/L_e)\,d\zeta = L_e^2/6$, so

$$w_1 = \frac{\hat M_0 L_e^2}{3EI} + \frac{\hat M_1 L_e^2}{6EI}, \qquad\text{i.e.}\qquad 3EI\,w_1 = \hat M_0 L_e^2 + \frac{\hat M_1 L_e^2}{2} \qquad \text{(NUBOW Eq. 4)}.$$

$\hat M_0$ weighs twice $\hat M_1$ in the deflection. A curvature near the clamp rotates the whole segment above it, so it acts on a long lever arm. A curvature near the free end has little beam left to move.

*Check.* With $\hat M_0 = \hat M_1 = \hat M$ (uniform moment), $w_1 = \hat M L_e^2/2EI$ and $\theta_1 = \hat M L_e/EI$, the cantilever under a constant moment.

*4. Chaining the segments.* Each segment i (length $L_i$) gives its own end deflection $w_i$ and slope $\theta_i$, relative to the tangent at its lower end. The deflection at the end of segment J adds the local deflections and the rigid displacement of each segment, which is carried by the rotations accumulated below it:

$$w(z_J) = \sum_{i=1}^{J} w_i + \sum_{i=2}^{J} L_i\sum_{j=1}^{i-1}\theta_j \quad \text{(NUBOW Eq. 6)}, \qquad \theta(z_J) = \sum_{i=1}^{J}\theta_i \quad \text{(NUBOW Eq. 7)}.$$

Equivalently, $w(z_J) = \sum_{i=1}^{J} w_i + \sum_{j=1}^{J-1}\theta_j\,(z_J - z_j)$, with $z_j$ the upper end of segment j. Eqs. 6–7 are illegible in the scanned report (only the sum limits, K and K − 1 in NUBOW's notation, are readable) and are written here from its text ("the sequential sum of deflections and displacements due to rotations of lower segments").

With $\hat M$ linear per segment, the trapezoid of Eqs. 4–5 is also exactly the load that the Hermite element integrates (§4.2). The VP1 check is in §4.1.

In [ ]:
# §1.6, NUBOW Eqs. 4-5: the trapezoid as two triangles, generic segment
Le, M0, M1 = 1.0, 1.0, 0.6
fig, a1 = plt.subplots(figsize=(8, 5))
zeta = np.array([0, Le])
a1.fill_between(zeta, [M0, 0], color="tab:blue", alpha=0.35, label=r"triangle $\hat M_0$: area $\hat M_0 L_e/2$")
a1.fill_between(zeta, [M0, 0], [M0, M1], color="tab:orange", alpha=0.35, label=r"triangle $\hat M_1$: area $\hat M_1 L_e/2$")
for zc, Mc, col, arm, cont in ((Le / 3, M0 / 3, "tab:blue", "2L_e/3", r"$\hat M_0 L_e^2/3$"),
                               (2 * Le / 3, (M0 + M1) / 3, "tab:orange", "L_e/3", r"$\hat M_1 L_e^2/6$")):
    a1.plot(zc, Mc, "o", color=col, ms=8)
    a1.annotate("", xy=(Le, Mc), xytext=(zc, Mc), arrowprops=dict(arrowstyle="<->", color=col))
    a1.text(0.5 * (zc + Le), Mc + 0.03, f"lever arm ${arm}$ → EI w₁ += " + cont, color=col, ha="center", fontsize=9)
a1.axvline(Le, color="k", lw=1)
a1.text(Le, -0.08, "upper end", ha="center", fontsize=9)
a1.set_xlabel(r"$\zeta$ from the lower end ($L_e$ = 1)"); a1.set_ylabel(r"$\hat M / EI$")
a1.set_title(r"generic segment. θ₁: areas (equal weights); w₁: areas × lever arms ($\hat M_0$ weighs twice)", fontsize=10)
a1.legend(fontsize=8, loc="upper right"); a1.set_ylim(-0.15, 1.15)
plt.tight_layout()

### 1.7 Pads: restrained bowing (future work)

Not treated quantitatively here. In a core, each wrapper carries load pads at a few heights, separated by small gaps from the pads of its neighbours or from a restraint ring. When the free bow closes a gap the pad touches and a contact force acts, which can only push. The problem is then nonlinear in the set of closed contacts, but linear once that set is fixed.

Possible implementations:
- the restrained deflection is the free bow of §1.4–1.6 plus the deflections caused by the contact forces (superposition, valid for a fixed set of closed contacts);
- Ma et al. (2019, Sect. 5) take rigid pads and cantilever deflections, and close the upper pad, then check the lower one (Fig. 13); an active set does the same for any number of pads (§2.3);
- NUBOW (ANL-8068, Sects. II.C–II.F) takes elastic pads (springs), unconstrained beam stiffness matrices for the whole row, and a relaxation iteration in which a tensile contact force is set to zero;
- EI, the gaps, the pad heights and the pad stiffness enter only here; the free bow does not depend on them.

Reference cases for a future check: IWGFR/75 Example 3 (one wrapper bowing into an array of neighbours, elastic pads, rotational spring at the base) and Joyo (Ohgama 2025).

## 2. Discretisation

### 2.1 Weak form, Hermite beam, linear rod

For one bending plane, with the free curvature $k = -\kappa^*_a$, the energy is $\frac12\int EI\,(u_a'' - k)^2 dz - \int q_a\,u_a\,dz$. Its variation gives the weak form

$$\int_0^H EI\,(u_a'' - k)\,\delta u_a''\,dz = \int_0^H q_a\,\delta u_a\,dz + \sum P\,\delta u_a + \sum M_c\,\delta u_a' ,$$

with P and $M_c$ the concentrated forces and moments at the nodes.

Only $u_a''$ appears, so the elements must share $u_a$ and $u_a'$ at the nodes (C¹). Each 2-node element of length $L_e$ therefore has the dofs $(u_1, \varphi_1, u_2, \varphi_2)$, $\varphi = u_a'$. These are four conditions, so the shape functions are the cubic Hermite polynomials ($\hat z = (z - z_1)/L_e$):

$$N_{1u} = 1 - 3\hat z^2 + 2\hat z^3, \qquad N_{1\varphi} = L_e\,(\hat z - 2\hat z^2 + \hat z^3), \qquad N_{2u} = 3\hat z^2 - 2\hat z^3, \qquad N_{2\varphi} = L_e\,(-\hat z^2 + \hat z^3),$$

$$\mathbf N'' = \frac{d^2\mathbf N}{dz^2} = \left(\frac{12\hat z - 6}{L_e^2},\ \frac{6\hat z - 4}{L_e},\ \frac{6 - 12\hat z}{L_e^2},\ \frac{6\hat z - 2}{L_e}\right).$$

Then

$$K_e = \int_0^{L_e} EI\,\mathbf N''\mathbf N''^T dz = \frac{EI}{L_e^3}\begin{bmatrix}12 & 6L_e & -12 & 6L_e\\ 6L_e & 4L_e^2 & -6L_e & 2L_e^2\\ -12 & -6L_e & 12 & -6L_e\\ 6L_e & 2L_e^2 & -6L_e & 4L_e^2\end{bmatrix}, \qquad f_e = \int_0^{L_e} EI\,k\,\mathbf N''\,dz + \int_0^{L_e} q_a\,\mathbf N\,dz .$$

The axial rod is linear and C⁰, as in `teaching/01_1D`. Its shape functions are $\mathbf N = (1 - \hat z,\ \hat z)$, and

$$K_e = \frac{EA}{L_e}\begin{bmatrix}1&-1\\-1&1\end{bmatrix}, \qquad f_e = \int_0^{L_e} EA\,\varepsilon_0^*\,\mathbf N'\,dz + \int_0^{L_e} q_z\,\mathbf N\,dz .$$

**Nodal exactness.**
- **Beam.** The Green's function of $(EI u'')''$ is cubic between the nodes, so it lies in the Hermite space. The nodal values of the Euler–Bernoulli element are then exact for any load integrated exactly (Reddy 1997). VP1 (§4) needs 3 elements, one per zone, to be exact everywhere: its w is cubic in the core and quadratic above.
- **Rod.** The rod is exact at the nodes and linear in between. Its N is exact at the element midpoints when $\varepsilon_0^*$ is linear on the element (VP1, §4.2); in general it is superconvergent there, O($L_e^2$).

In [ ]:
h, EI = 0.1, 5.3818e5
xi = np.linspace(0, 1, 200)
N, dN, _ = element.shape(xi, h)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4))
for i, (lab, sc) in enumerate(zip([r"$N_{1u}$", r"$N_{1\varphi}/L_e$", r"$N_{2u}$", r"$N_{2\varphi}/L_e$"], [1, 1 / h, 1, 1 / h])):
    a1.plot(xi, N[i] * sc, lw=2, label=lab)
a1.set_xlabel(r"$\hat z$"); a1.legend(ncol=4, fontsize=9); a1.set_title("cubic Hermite shape functions")
Nr, _ = element.rod_shape(xi, h)
a2.plot(xi, Nr[0], lw=2, label="$N_1$"); a2.plot(xi, Nr[1], lw=2, label="$N_2$")
a2.set_xlabel(r"$\hat z$"); a2.legend(); a2.set_title("linear rod shape functions")
plt.tight_layout()

K_q, K_cf = element.stiffness(EI, h), element.stiffness_closed(EI, h)
print("max |K_e(Gauss) - K_e(closed form, §2.1)| / max K =", np.max(np.abs(K_q - K_cf)) / np.max(np.abs(K_cf)))
H, P = 4.0, 1e3
U, R = solver.solve(element.stiffness(EI, H), np.array([0, 0, P, 0.0]), [0, 1])
print(f"1 element, tip force: w(H) = {U[2] * 1e3:.6f} mm, P H^3 / 3EI = {P * H**3 / (3 * EI) * 1e3:.6f} mm; "
      f"reactions {R} (N, N m)")

### 2.2 Ma's finite differences, and the clamp (Eqs. 28–29)

FADAC (Ma et al. 2019, Sect. 3.2, p. 31) divides the length into n equal steps $L_e = H/n$, with nodes $z_i = i\,L_e$, and solves $w'' = -M_{T,x}/EI = k$ by finite differences. In our notation:

$$\frac{w_{i-1} - 2w_i + w_{i+1}}{L_e^2} = k_i, \qquad i = 1,\dots,n-1 \qquad \text{(Ma Eq. 27)}.$$

That gives n − 1 equations for n unknowns $w_1 \dots w_n$ ($w_0 = 0$). The scheme has no rotation unknown, so the clamp condition $w'(0) = 0$ must be written with displacements:

$$w'(z_0) = \frac{-w_2 + 4w_1 - 3w_0}{2L_e} = 0 \quad \text{(Ma Eq. 28)}, \qquad\text{hence}\qquad 4w_1 = w_2 \quad \text{(Ma Eq. 29)}.$$

**Eq. (28) is the slope at $z_0$ of the parabola through the first three nodes**, the one-sided second-order difference. From Taylor:

$$w_1 = w_0 + L_e w_0' + \tfrac{L_e^2}{2} w_0'' + \tfrac{L_e^3}{6} w_0''', \qquad w_2 = w_0 + 2L_e w_0' + 2L_e^2 w_0'' + \tfrac{4L_e^3}{3} w_0''',$$

and $4w_1 - w_2$ cancels $w_0''$:

$$w_0' = \frac{-w_2 + 4w_1 - 3w_0}{2L_e} + \frac{L_e^2}{3}w_0''' .$$

With $w_0 = 0$ and $w'(0) = 0$ this is Eq. (29), the missing equation. Combined with Eq. (27) at i = 1 it gives $w_1 = L_e^2 k_1/2$, and the scheme then marches up: $w_{i+1} = 2w_i - w_{i-1} + L_e^2k_i$.

The truncation error of the central difference is $L_e^2 w''''/12$, so the scheme is O($L_e^2$). In the Hermite element the rotation is a dof: the clamp is imposed exactly (rotation dof of the clamp node set to 0), and the nodes are exact (§2.1).

The axial part is a first-order backward difference of $EA\,u_z' - N_T + f = 0$ (Ma Eq. 21, with f the gravity term):

$$EA\,\frac{u_{z,i} - u_{z,i-1}}{L_e} - N_T(z_i) + f(z_i) = 0 \quad \text{(Ma Eq. 25)}, \qquad u_{z,1} = \frac{L_e}{EA}\left[N_T(z_1) - f(z_1)\right] \quad \text{(Ma Eq. 26)}.$$

In [ ]:
# convergence on two loads with closed forms: VP1 (kinks on the nodes) and k = k0 (z/H)^2
Hv, k0v, zb, zt = 4.0, 6e-3, 1.5, 2.5
ramp = lambda z: np.clip((z - zb) / (zt - zb), 0, 1)
def w_vp1(z):
    z = np.asarray(z)
    return np.where(z <= zb, 0, np.where(z <= zt, k0v * (z - zb)**3 / 6, k0v * (1 / 6 + (z - zt) / 2 + (z - zt)**2 / 2)))
Hm, k0 = 3.056, 5e-3
cases = {"VP1 ramp": (Hv, lambda z: k0v * ramp(z), w_vp1(Hv)), "k0 (z/H)^2": (Hm, lambda z: k0 * (z / Hm)**2, k0 * Hm**2 / 12)}
fig, ax = plt.subplots(figsize=(7.5, 5))
for name, (H, k, wH) in cases.items():
    hs, e_fd, e_fe = [], [], []
    for N in (8, 16, 32, 64, 128, 256):
        x = np.linspace(0, H, N + 1)
        e_fd.append(abs(fdm.bending(x, k(x))[-1] - wH) / wH)
        zg, xg, wg = solver.gauss_points(x, 3)
        K, f = solver.assemble(x, 1.0, k(zg), xg, wg)
        e_fe.append(abs(solver.solve(K, f, [0, 1])[0][-2] - wH) / wH)
        hs.append(H / N)
    ax.loglog(hs, e_fd, "o-", label=f"Ma FDM, {name}: order {np.log2(e_fd[-2] / e_fd[-1]):.2f}")
    ax.loglog(hs, np.maximum(e_fe, 1e-17), "s--", label=f"Hermite FE, {name} (exact nodes)")
ax.set_xlabel(r"$L_e$ (m)"); ax.set_ylabel("relative error of w at the tip"); ax.legend(fontsize=8)
ax.set_title("Clamped cantilever, free curvature: FDM O(L_e²) vs Hermite FE")
plt.tight_layout()

### 2.3 Pads: active set (future work)

`solver.solve_contact` already implements a possible solution, which generalises Ma's flow chart (Fig. 13) to any number of pads: start with all pads open; solve with the closed pads imposed as displacements (exact, no penalty); close a pad that penetrates and release one whose reaction would pull; repeat until nothing changes. It is not validated in this notebook.

### 2.4 Round-off

The condition number of K grows as $n^4$ on a uniform mesh (fourth-order operator) and with the spread of element lengths: 1.3·10¹¹ for VP1 with nz = 400, where it leaves a round-off of 2·10⁻⁶ (URP) to 1.2·10⁻⁵ (core top) relative on w (§4.4); 5.5·10¹¹ for the 123 elements of the Ma case, which has a 2.7 mm element next to a station ($L_{e,\max}/L_{e,\min} \approx 10$; 3.8·10⁹ after diagonal scaling). Reactions come from $KU - f$, a difference of terms about 10⁴ times larger, so their round-off floor is about 10⁻¹⁰ of those terms (an estimate, TBC). The checks are scaled on them.

## 3. Practice

### 3.1 The section

The next cell builds the VP1 section (§1.5) and integrates the same quantities over the `bottom` facets of the Z3ST VP1 mesh (gmsh API).

In [ ]:
rows = []
for name, d in (("VP1", VP1),):
    c = case_mod.load_case(d)
    s, E = c.section, float(c.mat["E"])
    cf = s.closed_form()
    rows.append([name, f"{c.geo['D'] * 1e3:.3f}", f"{c.geo['t'] * 1e3:.1f}", f"{np.sqrt(3) * s.a_o * 1e3:.2f}",
                 f"{s.A:.6e}", f"{cf['A']:.6e}", f"{s.I:.6e}", f"{cf['I']:.6e}", f"{abs(s.I_xx - s.I_yy) / s.I:.1e}",
                 f"{s.radius_of_gyration() * 1e3:.2f}", f"{E * s.I:.4e}"])
table(rows, ["case", "D_o (mm)", "t (mm)", "outer F2F (mm)", "A_w (m²)", "closed form", "I_w (m⁴)", "closed form",
             "|I_xx − I_yy|/I_w", "r_g (mm)", "E I_w (N m²)"])

c = case_mod.load_case(VP1)
msh = section_from_msh(os.path.join(os.path.normpath(os.path.join(VP1, c.inp["case_3d"])), "mesh.msh"), "bottom")
s = c.section
print("VP1, beam section vs the 'bottom' facets of the Z3ST mesh:")
for k in ("A", "S_x", "S_y", "I_xx", "I_yy", "I_xy"):
    print(f"  {k:5s} beam {getattr(s, k):+.9e}   Z3ST mesh {msh[k]:+.9e}")

### 3.2 A case

A beam1d case is laid out as a z3st case, and `python -m beam1d <case>` writes `<case>/output/fields.npz`:
- **`geometry.yaml`**: `D`, `t`, `H`, `orientation`, `nz`, and `stations` (name → z). Stations play the role of the gmsh labels: each one becomes a node and a BC region.
- **`boundary_conditions.yaml`**:
  - `thermal`: T as a z3st `expression` in x, y, z, or as a `table` of the 12 key points;
  - `mechanical`: `Dirichlet` (`displacement: [u_x, u_y, u_z]`, `rotation: [u_x', u_y']`), `Neumann` (force, moment at a station), `Contact` (pads, for future restrained cases), `Body` (gravity).
- **`input.yaml`**: paths, `materials` (a z3st card), `models`, quadrature, contact iterations; `reference` (paper curves) or `case_3d` (Z3ST run) for the comparison scripts.

`Allrun` runs the solve, `non-regression.py` (analytic checks, z3st `finish`, tolerance 1e-8) and the comparison script.

In [ ]:
for f in ("geometry.yaml", "boundary_conditions.yaml", "input.yaml"):
    print(f"# ---- VP1/{f}\n" + open(os.path.join(VP1, f)).read())

## 4. IAEA VP1

IWGFR/75 Example 1 (VP1; specification in Part I, App. 2, pp. 107–108), in three steps:
- §4.1–4.2: solved with the standard beam theory of §1, in closed form and by FEM;
- §4.3: compared with the IAEA document (linear theory and the results of the participants);
- §4.4: compared with the Z3ST 3D run of the same case.

### 4.1 Standard theory: closed form, step by step

A cantilever beam with a thermal eigenstrain (§1.3–1.4), clamped at the support.

**Hypotheses** (first approximation, as IWGFR/75 Part I, §§15–16, pp. 12–13):
- Euler–Bernoulli: plane sections stay plane and normal to the axis, no shear (§1.1);
- small displacements and strains, linear elastic;
- E and α constant: uniform over the section and along z, independent of T;
- the section is the undeformed wall Ω₀ at every z: no axial elongation of the geometry, no lateral dilation, no feedback of the deformation on T;
- clamp at z = 0, free top, no other load.

**Data** (SI; a T difference in °C equals one in K, so α = 18e-6 1/°C = 1.8e-5 1/K):

| symbol | value | meaning |
|---|---|---|
| $a_m$ | 0.075 m | mid-wall corner radius, $(a_o + a_i)/2$ (§1.5) |
| $D_m$ | $2a_m$ = 0.150 m | mid-wall corner-to-corner distance, the direction of ΔT (IWGFR: "distance between the mid-walls of the corners") |
| t | 0.003 m | wall thickness |
| H | 4.0 m | height; LRP at 3.0 m, URP at H |
| $z_b$, $z_t$, $L_c$ | 1.5, 2.5, 1.0 m | core bottom, core top, core length |
| E | 1.7e5 MPa = 1.7e11 Pa | Young's modulus |
| α | 1.8e-5 1/K | expansion coefficient |
| $T_\mathrm{in}$ | 400 °C = 673.15 K | inlet T, whole section below the core |
| $T_\mathrm{ref}$ | 673.15 K | stress-free T of the card `15_15Ti_const.yaml`; IWGFR measures lengths from 20 °C = 293.15 K (step 5) |

**Step 0: temperature.** With the core ramp $\eta(z) = \min(\max((z - z_b)/L_c, 0), 1)$ (−),

$$T(y, z) = T_\mathrm{in} + \eta(z)\left[\Delta T_m - \frac{\Delta T}{D_m}\,y\right] \quad (\mathrm K), \qquad \Delta T_m = 125\ \mathrm K, \quad \Delta T = 50\ \mathrm K .$$

- $\Delta T_m = \bar T - T_\mathrm{in}$ (K): rise of the section-mean T over the inlet above the core, 525 − 400 °C = 125 K (the T at the centroid, y = 0);
- $\Delta T = T_\mathrm{hot} - T_\mathrm{cold}$ (K): difference between the hot and cold corners above the core, 550 − 500 °C = 50 K, taken over $D_m$.

At the core top this gives:
- 500 °C at the cold corner (y = +D_m/2) and 550 °C at the hot corner (y = −D_m/2);
- 512.5 / 537.5 °C at the intermediate corners (y = ±D_m/4);
- 525 °C at the centroid.

$\Delta T/D_m$ = 333.3 K/m is the `25.0 / 0.075` of `VP1/boundary_conditions.yaml`. T does not depend on x, and it is linear in y at every z.

**Step 1: section.** The hexagonal annulus has $a_o - a_i = 2t/\sqrt3$ and $a_o + a_i = 2a_m$, so $a_o^2 - a_i^2 = 4a_m t/\sqrt3$ and $a_o^2 + a_i^2 = 2a_m^2 + 2t^2/3$. The closed forms of §1.5 become

$$A_w = \tfrac{3\sqrt3}{2}(a_o^2 - a_i^2) = 6\,a_m t, \qquad I_w = \tfrac{5\sqrt3}{16}(a_o^4 - a_i^4) = \tfrac52\,a_m^3 t\left(1 + \frac{t^2}{3a_m^2}\right),$$

both exact for sharp corners. Numerically:
- $A_w$ = 1.350e-3 m² and $I_w$ = 3.16575e-6 m⁴; the thin-wall $\tfrac52 a_m^3 t$ is 0.05 % lower;
- EA = 2.2950e8 N and EI = 5.3818e5 N m²;
- by symmetry, $S_x = S_y = 0$ and $I_{xy} = 0$.

**Step 2: equivalent thermal force and moment.** With E and α constant they come out of the integrals:

$$N_T(z) = E\alpha\int_A (T - T_\mathrm{ref})\,dA = E\alpha\,\eta\left[\Delta T_m A_w - \frac{\Delta T}{D_m}\int_A y\,dA\right] = EA\,\alpha\,\eta\,\Delta T_m \quad (\mathrm N),$$

$$M_{T,y}(z) = E\alpha\int_A (T - T_\mathrm{ref})\,y\,dA = E\alpha\,\eta\left[\Delta T_m\int_A y\,dA - \frac{\Delta T}{D_m}\int_A y^2\,dA\right] = -EI\,\frac{\alpha\,\Delta T}{D_m}\,\eta \quad (\mathrm{N\,m}),$$

since $\int_A y\,dA = 0$ and $\int_A y^2\,dA = I_w$, and $M_{T,x} = 0$.

$T_\mathrm{ref}$ drops out of $M_T$, because $\int y\,dA = 0$, but not out of $N_T$.

Above the core (η = 1):
- $N_T = 2.2950\text{e}8 \cdot 1.8\text{e-}5 \cdot 125 = 5.164\text{e}5$ N;
- $|M_{T,y}| = 5.3818\text{e}5 \cdot 6.0\text{e-}3 = 3.229\text{e}3$ N m.

**Step 3: free strains (§1.3).**

$$\varepsilon_0^* = \frac{N_T}{EA} = \alpha\,\Delta T_m\,\eta = 2.25\times10^{-3}\,\eta \;\;(-), \qquad \kappa^*_y = \frac{M_{T,y}}{EI} = -k_0\,\eta, \quad k_0 = \frac{\alpha\,\Delta T}{D_m} = \frac{1.8\times10^{-5}\cdot 50}{0.150} = 6.0\times10^{-3}\ \mathrm{1/m} .$$

E and $I_w$ drop out: the free bow depends on α, ΔT and $D_m$ only.

**Step 4: equilibrium of the cantilever (§1.4 step 4).** There is no load and the top is free:
- $N(H) = 0$ and $N' = 0$ give N ≡ 0;
- $m(H) = m'(H) = 0$ and $m'' = 0$ give m ≡ 0.

From §1.3:

$$N = EA\,(u_z' - \varepsilon_0^*) = 0 \;\Rightarrow\; u_z' = \alpha\,\Delta T_m\,\eta, \qquad m_y = EI\,(u_y'' + \kappa^*_y) = 0 \;\Rightarrow\; u_y'' = k_0\,\eta ,$$

with $u_z(0) = 0$ and $u_y(0) = u_y'(0) = 0$. The sign is positive: the hot side (−y) lengthens and becomes convex, so the axis bows towards the cold corner (+y).

**Step 5: axial displacement.** The first and second integrals of the ramp, from the clamp, are

$$R_1(z) = \int_0^z \eta\,d\zeta = \begin{cases} 0 & z \le z_b \\ \dfrac{(z - z_b)^2}{2L_c} & z_b < z \le z_t \\ \dfrac{L_c}{2} + z - z_t & z > z_t \end{cases} \quad (\mathrm m), \qquad R_2(z) = \int_0^z R_1\,d\zeta = \begin{cases} 0 \\ \dfrac{(z - z_b)^3}{6L_c} \\ \dfrac{L_c^2}{6} + \dfrac{L_c\,(z - z_t)}{2} + \dfrac{(z - z_t)^2}{2} \end{cases} \quad (\mathrm{m^2}),$$

so that

$$u_z(z) = \alpha\,\Delta T_m\,R_1(z) \quad (\mathrm m).$$

This gives $u_z$ = 1.125, 2.250 and 4.500 mm at the core top, LRP and URP. IWGFR measures the lengths from 20 °C, so the whole wall also carries $\alpha(T_\mathrm{in} - 293.15\ \mathrm K) = 6.84\times10^{-3}$, and

$$u_z = \alpha\left[(T_\mathrm{in} - 293.15)\,z + \Delta T_m R_1(z)\right] = 18.225,\ 22.770,\ 31.860\ \mathrm{mm}.$$

The bow does not change.

**Step 6: slope and deflection.**

$$\theta(z) = u_y' = k_0\,R_1(z) \;\;(\mathrm{rad}), \qquad w(z) = u_y = k_0\,R_2(z) \;\;(\mathrm m).$$

At $z_t$, $R_2$, $R_1$ and η are continuous ($L_c^2/6$, $L_c/2$, 1), so w is C²; only w‴ jumps, at the kinks of $\kappa^*_y$.

Cauchy's formula (§1.4 step 4) gives the same tip value; it is split segment by segment after the compatibility figure.

| station | z (m) | η | θ (mrad) | w (mm) | $u_z$, $T_\mathrm{ref}$ 400 °C (mm) | $u_z$, from 20 °C (mm) |
|---|---|---|---|---|---|---|
| core top | 2.5 | 1 | 3.0 | 1.000 | 1.125 | 18.225 |
| LRP | 3.0 | 1 | 6.0 | 3.250 | 2.250 | 22.770 |
| URP | 4.0 | 1 | 12.0 | 12.250 | 4.500 | 31.860 |

**Step 7: stresses and reactions.** The residual strain (§1.4 step 2) is

$$\varepsilon_r = \alpha(T - T_\mathrm{ref}) - \varepsilon_0^* - \kappa^*_y\,y = \alpha \eta\left[\Delta T_m - \frac{\Delta T}{D_m}y\right] - \alpha \eta\,\Delta T_m + \alpha \eta\,\frac{\Delta T}{D_m}\,y = 0 .$$

So $\sigma_{zz} \equiv 0$, N ≡ m ≡ 0, and the clamp reactions are zero: VP1 is a free expansion. EI enters only through restraints (Example 3, pads).

For scale, a fully restrained beam (§1.4 step 5) above the core would carry:
- $N = -N_T$ = −516 kN and $m_y = M_{T,y}$;
- $\sigma_{zz} = -E\alpha(T - T_\mathrm{ref})$ = −459 MPa at the hot corner (150 K) and −306 MPa at the cold one (100 K).

**What the closed form leaves out:**
- second-order effects: axial elongation and lateral dilation of the section (their size in §4.3);
- the thermal shear of the core, $\gamma = -c\,k'$: −13.8 µm in Z3ST (§4.4);
- α(T) and E(T) (§1.4, case b);
- the in-plane stresses.

The next cell evaluates every step. It reads E, α, $T_\mathrm{ref}$, $a_m$, t, H and the stations from the case files. $T_\mathrm{in}$, $\Delta T_m$ and ΔT are the parameters of the T string, and the section quadrature of `thermal.build`, which reads that string, checks them through $\varepsilon_0^*$ and $\kappa^*_y$.

In [ ]:
# VP1 closed form (§4.1), SI units, T differences in K
c = case_mod.load_case(VP1)
s = c.section
E, alpha, T_ref = float(c.mat["E"]), float(c.mat["alpha"]), float(c.mat["T_ref"])
T_in, dT_m, dT, D = 673.15, 125.0, 50.0, 2 * s.a_m        # (K), (K), (K), D_m (m)
assert abs(T_ref - T_in) < 1e-9                           # the closed form takes T_ref = T_in
zb, zt, H = c.stations["core_bottom"], c.stations["core_top"], c.H
L_c = zt - zb

# step 1: section
A = 6 * s.a_m * s.t
I = 2.5 * s.a_m**3 * s.t * (1 + s.t**2 / (3 * s.a_m**2))
print(f"A = {A:.6e} m^2 (quadrature {s.A:.6e}), I = {I:.6e} m^4 (quadrature {s.I:.6e}), "
      f"thin wall 5/2 a_m^3 t = {2.5 * s.a_m**3 * s.t:.6e} m^4")
print(f"EA = {E * A:.4e} N, EI = {E * I:.4e} N m^2")

# steps 2-3: N_T, M_T, free strains, against the section quadrature of the T string
r = lambda z: np.clip((z - zb) / L_c, 0.0, 1.0)
k0 = alpha * dT / D
N_T = lambda z: E * A * alpha * dT_m * r(z)
M_T = lambda z: -E * I * k0 * r(z)
_, eps = thermal.build(c, s)
zq = np.linspace(0, H, 81)
print(f"k0 = alpha dT / D = {k0:.6e} 1/m, N_T(top) = {N_T(H):.4e} N, M_T,y(top) = {M_T(H):.4e} N m")
print(f"max |eps0* - alpha dT_m r| = {np.abs(s.mean(eps, zq) - alpha * dT_m * r(zq)).max():.1e}, "
      f"max |kappa*_y + k0 r| = {np.abs(s.curvature(eps, zq)[:, 1] + k0 * r(zq)).max():.1e} 1/m, "
      f"-kappa*_y(H) D_m / (alpha dT) = {-s.curvature(eps, [H])[0, 1] * D / (alpha * dT):.6f}")   # D_m = 2 a_m, 0.150 m to relative 4e-9

# steps 5-6: R1, R2, then u_z, theta, w
R1 = lambda z: np.where(z <= zb, 0, np.where(z <= zt, (z - zb)**2 / (2 * L_c), L_c / 2 + z - zt))
R2 = lambda z: np.where(z <= zb, 0, np.where(z <= zt, (z - zb)**3 / (6 * L_c),
                                              L_c**2 / 6 + L_c * (z - zt) / 2 + (z - zt)**2 / 2))
rows = []
for name, z in (("core top", zt), ("LRP", c.stations["LRP"]), ("URP", H)):
    uz20 = alpha * ((T_in - 293.15) * z + dT_m * R1(z))
    rows.append([name, z, f"{k0 * R1(z) * 1e3:.3f}", f"{k0 * R2(z) * 1e3:.4f}",
                 f"{alpha * dT_m * R1(z) * 1e3:.4f}", f"{uz20 * 1e3:.4f}"])
table(rows, ["station", "z (m)", "θ (mrad)", "w (mm)", "u_z, T_ref 400 °C (mm)", "u_z, T_ref 20 °C (mm)"])

# Cauchy's formula at the tip, by quadrature, against k0 R2(H)
zz = np.linspace(0, H, 400001)
print(f"w(H): Cauchy {np.trapezoid((H - zz) * k0 * r(zz), zz) * 1e3:.6f} mm, k0 R2(H) = {k0 * R2(H) * 1e3:.6f} mm")

# step 7: residual strain of the T string (zero: T linear in y)
z_pts = np.array([zb + 0.3, zt, H])
eps_r = s.field(eps, z_pts) - s.mean(eps, z_pts)[None, :] - s.xy[:, 1][:, None] * s.curvature(eps, z_pts)[:, 1][None, :]
print(f"max |eps_r| = {np.abs(eps_r).max():.1e}; fully restrained above the core: "
      f"sigma_zz = {-E * alpha * (dT_m + dT / 2) / 1e6:.0f} MPa (hot corner), {-E * alpha * (dT_m - dT / 2) / 1e6:.0f} MPa (cold)")

**Compatibility of the VP1 field in 3D (§1.4).** For an isotropic eigenstrain a continuous body follows ε* without stress only where every second derivative of ε* is zero, i.e. ε* linear in (x, y, z). With α constant, $\varepsilon^* = \alpha\,\eta(z)\,[\Delta T_m - (\Delta T/D_m)\,y]$ (step 0, $T_\mathrm{ref} = T_\mathrm{in}$) is linear in y and independent of x, so only the derivatives with z can be non-zero:

| derivative | expression | non-zero where |
|---|---|---|
| $\varepsilon^*_{,yz}$ | $-\alpha\,(\Delta T/D_m)\,\eta' = -\alpha\,\Delta T/(D_m L_c)$ = −6.0e-3 1/m² | the whole core, $z_b < z < z_t$ |
| $\varepsilon^*_{,zz}$ | $\alpha\,\eta''\,[\Delta T_m - (\Delta T/D_m)\,y]$ | the planes $z_b$ and $z_t$ only |

- **Core, distributed.** $\varepsilon^*_{,yz} = \kappa^{*\prime}_y \neq 0$: $\boldsymbol\kappa^*$ varies along z, and the 3D wall shears ($\gamma$, *What the closed form leaves out*).
- **Ramp ends, local.** η is piecewise linear: η′ is a step (0, $1/L_c$, 0), and η″ is a pair of Dirac deltas, $\eta'' = [\delta(z - z_b) - \delta(z - z_t)]/L_c$. So $\varepsilon^*_{,zz}$ is a Dirac layer on each plane: zero elsewhere, infinite on the plane, with a finite integral across it. That integral is the jump in the slope of ε* along z,

$$[\varepsilon^*_{,z}] = \pm\frac{\alpha}{L_c}\left[\Delta T_m - \frac{\Delta T}{D_m}\,y\right] \quad (\mathrm{1/m}),$$

+ at $z_b$ and − at $z_t$: 2.70e-3 1/m at the hot corner (150 K), 2.25e-3 at the centroid (125 K), 1.80e-3 at the cold corner (100 K).
  - The $\Delta T_m$ term alone is incompatible on these two planes only. It turns a hexagonal prism (below) into a frustum of a hexagonal pyramid (core) and a larger prism (above). Each zone is compatible, but the linear-ε* displacement fields of §1.4 have different gradients $\mathbf b$ and do not join on the plane, so the wall bends locally there.
  - The ΔT term is incompatible on the two planes as well, in addition to its distributed part in the core.
- **Below $z_b$ and above $z_t$.** η is constant and ε* is linear in (x, y, z), so ε* is locally compatible there. The self-stress of the core decays with distance from $z_b$ and $z_t$. At the clamp, 1.5 m (10 $D_m$) below the core, ε* = 0 ($T = T_\mathrm{in} = T_\mathrm{ref}$). The 3D clamp fixes the whole bottom face, so it would restrain any lateral expansion, but there is none, and the clamp adds no stress.

In [ ]:
# §4.1, compatibility of the VP1 field (§1.4) along z (left: T on the y-z plane; centre: eps*,yz and the layers of eps*,zz; right: the bow)
c = case_mod.load_case(VP1)
s = c.section
zb, zt, H = c.stations["core_bottom"], c.stations["core_top"], c.H
alpha, dT, D = float(c.mat["alpha"]), 50.0, 2 * s.a_m
L_c, k0 = zt - zb, alpha * dT / D
T_fun, eps = thermal.build(c, s)
z = np.linspace(0, H, 801)
yy = np.linspace(-D / 2, D / 2, 61)
Yg, Zg = np.meshgrid(yy, z)
e_yz = np.gradient(s.curvature(eps, z)[:, 1], z)          # T linear in y: kappa*_y = eps*,y, so d kappa*_y/dz = eps*,yz
w = k0 * R2(z)
fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(15, 7), sharey=True)
pc = a1.pcolormesh(Yg * 1e3, Zg, T_fun((0 * Yg, Yg, Zg)) - 273.15, cmap="inferno", shading="auto")
a1.contour(Yg * 1e3, Zg, T_fun((0 * Yg, Yg, Zg)) - 273.15, levels=np.arange(410, 550, 10), colors="w", linewidths=0.6)
fig.colorbar(pc, ax=a1, label="T (°C)")
a1.set_xlabel("y (mm): hot corner −y, cold corner +y"); a1.set_ylabel("z (m)")
a1.set_title("T(y, z), independent of x. Isotherms: none or equally spaced\nparallel lines = compatible; fanning in the core = incompatible", fontsize=10)
a2.plot(e_yz * 1e3, z, "tab:orange", lw=2, label=r"$\varepsilon^*_{,yz}$ (numerical)")
a2.axvline(-alpha * dT / (D * L_c) * 1e3, color="k", ls=":", lw=1, label=r"$-\alpha\Delta T/(D_m L_c)$")
for zk, sgn in ((zb, 1), (zt, -1)):
    a2.annotate("", xy=(0, zk), xytext=(sgn * 5, zk), arrowprops=dict(arrowstyle="<-", color="crimson", lw=2))
    a2.text(sgn * 5, zk + 0.06, r"$\delta$ layer of $\varepsilon^*_{,zz}$" + (" (+)" if sgn > 0 else " (−)") + ", schematic",
            color="crimson", ha="center", fontsize=9)
a2.set_xlim(-9, 9); a2.set_xlabel(r"$\varepsilon^*_{,yz}$ ($10^{-3}$ 1/m²)"); a2.legend(fontsize=8, loc="upper left")
a2.set_title("second derivatives of ε*", fontsize=10)
a3.plot(w * 1e3, z, "k", lw=2)
a3.set_xlabel("w (mm), towards the cold corner (+y)"); a3.set_title("free bow w = k₀ R₂(z) (§4.1)", fontsize=10)
zones = [(0, zb, "tab:green", "compatible\nε* = 0"), (zb, zt, "tab:orange", "ε*,yz ≠ 0: distributed,\n3D wall shears (γ)"),
         (zt, H, "tab:green", "compatible\nε* linear in y")]
for a in (a1, a2, a3):
    for z0, z1, col, _ in zones:
        a.axhspan(z0, z1, color=col, alpha=0.08 if a is not a1 else 0.0)
    for zk in (zb, zt):
        a.axhline(zk, color="crimson", lw=1, ls="--")
for z0, z1, col, lab in zones:
    a3.text(0.97, 0.5 * (z0 + z1), lab, transform=a3.get_yaxis_transform(), ha="right", va="center", fontsize=9, color=col)
a1.set_ylim(0, H)
print(f"core: eps*,yz = {np.median(e_yz[(z > zb + 0.05) & (z < zt - 0.05)]):.4e} 1/m^2, "
      f"-alpha dT / (D_m L_c) = {-alpha * dT / (D * L_c):.4e} 1/m^2")
plt.tight_layout()

How to read the figure:
- left: T on the plane x = 0, hollow interior included. The isotherms are absent (below $z_b$) or parallel and equally spaced (above $z_t$) where ε* is compatible, and fan out in the core;
- centre: the orange curve is $d\kappa^*_y/dz$ from the section quadrature, equal to $\varepsilon^*_{,yz}$ because T is linear in y, on the closed form (printed). The red arrows mark the Dirac layers of $\varepsilon^*_{,zz}$ with their sign; a delta has no value on this axis (its weight is in 1/m), so their length is schematic;
- right: the beam bow $w = u_y$. The core incompatibility appears only as $w''' = k_0/L_c$: Euler–Bernoulli has no shear, so the beam follows it without stress. The Dirac layers do not appear at all, since $w'' = k_0\,\eta$ is continuous. Their self-stresses (local $\sigma_{zz}$ and hoop bending at $z_b$ and $z_t$, shear in the core) need a 3D model: a plane section cannot represent them.

**Segment by segment on VP1 (§1.6).** The axis curvature is $k(\zeta) = u_y''(\zeta) = -\kappa^*_y(\zeta) = k_0\,\eta(\zeta)$ (1/m), with ζ the integration variable along z (global here, not the local ζ of §1.6). The segments are [0, $z_b$] (k = 0), the core [$z_b$, $z_t$] (k from 0 to $k_0$; $\hat M_0 = 0$, $\hat M_1 = EIk_0$) and [$z_t$, H] ($k = k_0$, $L_3 = H - z_t$ = 1.5 m).

Cauchy's formula $w(H) = \int_0^H (H - \zeta)\,k(\zeta)\,d\zeta$, split over the segments with the core lever arm written as $H - \zeta = (z_t - \zeta) + L_3$:

$$w(H) = \underbrace{\int_{z_b}^{z_t}(z_t - \zeta)\,k\,d\zeta}_{w_2} + L_3\underbrace{\int_{z_b}^{z_t}k\,d\zeta}_{\theta_2} + \underbrace{\int_{z_t}^{H}(H - \zeta)\,k_0\,d\zeta}_{w_3} .$$

This is NUBOW Eq. 6 with J = 3 and $w_1 = \theta_1 = 0$: the local deflection of each segment, plus the rigid carry of the core rotation over the segment above. The split is exact; the approximations are the hypotheses of the closed form, small displacements among them ($L\sin\theta \approx L\theta$; θ ≤ 12 mrad here).

| term | formula | value |
|---|---|---|
| $w_2$ | $k_0L_c^2/6$ | 1.000 mm |
| $L_3\theta_2$ | $L_3\cdot k_0L_c/2$ | 1.5 m × 3.0 mrad = 4.500 mm |
| $w_3$ | $k_0L_3^2/2$ | 6.750 mm |
| **w(H)** | | **12.250 mm**, as in step 6 |

$\theta(H) = \theta_2 + k_0L_3$ = 3.0 + 9.0 = 12.0 mrad. $L_3\theta_2$ is the term most easily missed: the local deflections alone, $w_2 + w_3$ = 7.75 mm, lose more than a third of the tip bow.

In [ ]:
# §4.1, segment by segment on VP1 (§1.6): the tip bow as the sum of the segment bows and the rigid carry of the core rotation. Needs the VP1 closed-form cell of §4.1 (c, alpha, dT, D, R2)
fig, a2 = plt.subplots(figsize=(8, 5))
zb, zt, H = c.stations["core_bottom"], c.stations["core_top"], c.H
k0, L_c, L3 = alpha * dT / D, zt - zb, H - zt
z = np.linspace(0, H, 400)
w = k0 * R2(z)
w2, th2, w3 = k0 * L_c**2 / 6, k0 * L_c / 2, k0 * L3**2 / 2
a2.plot(z, w * 1e3, "k", lw=2, label="w(z)")
zr = np.linspace(zt, H, 50)
a2.plot(zr, (w2 + th2 * (zr - zt)) * 1e3, "tab:blue", ls="--", label=r"core end carried rigidly: $w_2 + \theta_2 (z - z_t)$")
for y0, y1, col, lab in ((0, w2, "tab:green", f"w₂ = {w2 * 1e3:.3f}"), (w2, w2 + L3 * th2, "tab:blue", f"L₃θ₂ = {L3 * th2 * 1e3:.3f}"),
                         (w2 + L3 * th2, w2 + L3 * th2 + w3, "tab:red", f"w₃ = {w3 * 1e3:.3f}")):
    a2.annotate("", xy=(H + 0.05, y1 * 1e3), xytext=(H + 0.05, y0 * 1e3), arrowprops=dict(arrowstyle="<->", color=col, lw=2))
    a2.text(H + 0.1, 0.5 * (y0 + y1) * 1e3, lab + " mm", color=col, va="center", fontsize=9)
for zk in (zb, zt):
    a2.axvline(zk, color="0.6", lw=1, ls=":")
a2.set_xlim(0, H + 0.9); a2.set_xlabel("z (m)"); a2.set_ylabel("w (mm)")
a2.set_title(f"VP1 tip: w(H) = w₂ + L₃θ₂ + w₃ = {(w2 + L3 * th2 + w3) * 1e3:.3f} mm", fontsize=10)
a2.legend(fontsize=8, loc="upper left")
plt.tight_layout()

### 4.2 Discretisation: FEM under the VP1 hypotheses

The hypotheses are those of §4.1: Euler–Bernoulli, small strains, E and α constant, undeformed section, clamp at z = 0. The bending and axial problems are solved separately, because $\mathbf B_1 = 0$ (§1.3).

**1. beam1d, bending: Hermite Euler–Bernoulli element (§2.1).** Input: the T string, integrated over the section. In the y plane, with $k = -\kappa^*_y = k_0\,\eta$ and $\hat M = EI\,k = -M_{T,y}$ (N m), the weak form is

$$\int_0^H EI\,u_y''\,\delta u_y''\,dz = \int_0^H \hat M\,\delta u_y''\,dz .$$

$\hat M$ is linear on each element, between the nodal values $\hat M_a$ and $\hat M_b$. This is the trapezoid of §1.6, $\hat M = \hat M_0 + (\hat M_1 - \hat M_0)\,\zeta/L_e$. With $\mathbf N''$ of §2.1, integrating $\hat M\,\mathbf N''$ over the element gives the consistent load on $(u_1, \varphi_1, u_2, \varphi_2)$:

$$f_e = \left(\frac{\hat M_b - \hat M_a}{L_e},\ -\hat M_a,\ \frac{\hat M_a - \hat M_b}{L_e},\ \hat M_b\right) .$$

$\hat M$ is continuous, so the nodal moments cancel at every interior node, and the nodal forces add up to the jump of $\hat M'$. The VP1 thermal load is therefore exactly equivalent to three concentrated loads (the strong form, $(EI u_y'')'' = \hat M''$):
- a force $+EI k_0/L_c$ = +3229 N at $z_b$;
- a force $-EI k_0/L_c$ = −3229 N at $z_t$;
- a moment $EI k_0$ = 3229 N m at the top.

Superposing the cantilever solutions for these three loads, $P z_P^2(3H - z_P)/6EI$ for a force P at $z_P$ and $M_c H^2/2EI$ for a tip moment $M_c$, gives

$$w(H) = k_0\left[\frac{H^2}{2} + \frac{z_b^2(3H - z_b)}{6L_c} - \frac{z_t^2(3H - z_t)}{6L_c}\right] = 6\times10^{-3}\,[8 + 3.9375 - 9.8958] = 12.250\ \mathrm{mm},$$

which is the closed form of §4.1.
- **Quadrature.** $\hat M\,\mathbf N''$ is quadratic on each element, so 2 Gauss points are exact; `input.yaml` uses 3.
- **Exactness.** With the kinks $z_b$, $z_t$ on nodes (they are stations), Gauss quadrature integrates $f_e$ exactly, so the nodes are exact (§2.1). w is then cubic on each zone, so it is exact between the nodes as well; the minimal mesh needs 3 elements. $u_y''$ is linear on each element and equals k, so $m_y = EI(u_y'' - k)$ and $\sigma_{zz}$ vanish to round-off.

**2. beam1d, axial: linear rod (§2.1).** Input: the same T string. The weak form is $\int EA\,u_z'\,\delta u_z'\,dz = \int EA\,\varepsilon_0^*\,\delta u_z'\,dz$, and the element load is $f_e = EA\,\bar\varepsilon_{0,e}^*\,(-1, +1)$, with $\bar\varepsilon_{0,e}^*$ the element mean of $\varepsilon_0^*$.
- **Nodes.** The nodes are exact: the Green's function of the rod is piecewise linear.
- **Between the nodes.** $u_z$ is quadratic in the core, so the linear interpolation is off by at most $L_e^2|u_z''|/8 = L_e^2\alpha\Delta T_m/(8L_c)$, i.e. 2.8 µm for $L_e$ = 0.1 m and 0.03 µm for $L_e$ = 0.01 m.
- **Axial force.** $u_z'$ is constant per element and equals the element mean of $\varepsilon_0^*$, so $N = EA(u_z' - \varepsilon_0^*)$ is zero at the midpoints of the core elements and identically zero elsewhere. At the ends of a core element it is $\pm EA\,\alpha\Delta T_m L_e/(2L_c)$ = ±25.8 kN for $L_e$ = 0.1 m (±2.6 kN for $L_e$ = 0.01 m). This is an artefact of the C⁰ interpolation, not a force, so N is read at the midpoints.

**3. Central finite differences (§2.2).** Input: the curvature k. This scheme is not a FEM: it is a central difference on $w'' = k$, of order O($L_e^2$). Its error at the URP for VP1 is in the convergence figure of §2.2.

The next two cells check the equivalent loads and the exactness on the minimal mesh (nodes 0, $z_b$, $z_t$, H), and the rod errors on the mesh of `VP1/geometry.yaml` (`nz` elements plus the stations).

In [ ]:
# §4.2: the VP1 thermal moment as consistent nodal loads on the minimal mesh (nodes 0, z_b, z_t, H). Needs the VP1 closed-form cell (§4.1)
zb, zt, H = c.stations["core_bottom"], c.stations["core_top"], c.H
L_c, k0, EI = zt - zb, alpha * dT / D, E * I
zn = np.array([0.0, zb, zt, H])
zg, xg, wg = solver.gauss_points(zn, 3)
K, f = solver.assemble(zn, EI, k0 * np.clip((zg - zb) / L_c, 0, 1), xg, wg)
U, R = solver.solve(K, f, [0, 1])
z = np.linspace(0, H, 400)
w_cf = k0 * R2(z)
fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(15, 6.5), sharey=True)
a1.fill_betweenx(z, EI * k0 * np.clip((z - zb) / L_c, 0, 1) / 1e3, color="tab:purple", alpha=0.3)
a1.plot(EI * k0 * np.clip((z - zb) / L_c, 0, 1) / 1e3, z, "tab:purple", lw=2)
a1.plot(EI * k0 * np.clip((zn - zb) / L_c, 0, 1) / 1e3, zn, "ko")
a1.set_xlabel(r"$\hat M = EI\,k = -M_{T,y}$ (kN m)"); a1.set_ylabel("z (m)")
a1.set_title(r"thermal moment, linear per element", fontsize=10)
a2.plot([0, 0], [0, H], "k", lw=4, solid_capstyle="butt")
a2.fill_between([-0.6, 0.6], -0.12, 0, color="0.5", hatch="///", alpha=0.5)
for i, zi in enumerate(zn):
    Fi, Mi = f[2 * i], f[2 * i + 1]
    if abs(Fi) > 1e-6 * EI * k0:
        a2.annotate("", xy=(0, zi), xytext=(-0.5 * np.sign(Fi), zi), arrowprops=dict(arrowstyle="->", color="tab:red", lw=2.5))
        a2.text(-0.5 * np.sign(Fi), zi + 0.08, f"F = {Fi:+.0f} N", color="tab:red", ha="center", fontsize=9)
    if abs(Mi) > 1e-6 * EI * k0:
        a2.annotate("", xy=(0.25, zi + 0.1), xytext=(-0.25, zi + 0.1),
                    arrowprops=dict(arrowstyle="->", color="tab:blue", lw=2, connectionstyle=f"arc3,rad={-0.6 * np.sign(Mi)}"))
        a2.text(0, zi + 0.42, f"M = {Mi:+.0f} N m", color="tab:blue", ha="center", fontsize=9)
a2.text(0, -0.3, f"clamp: reactions {R[0]:+.1e} N, {R[1]:+.1e} N m", ha="center", fontsize=8)
a2.set_xlim(-1, 1); a2.set_xticks([]); a2.set_ylim(-0.4, H + 0.6); a2.set_xlabel("→ +y (cold corner)")
a2.set_title(r"$f = \sum_e \int \hat M\,\mathbf{N}''\,dz$: moments cancel at interior nodes", fontsize=10)
a3.plot(w_cf * 1e3, z, "k", lw=2, label="closed form (§4.1)")
wq, _, _ = solver.evaluate(zn, U, z)
a3.plot(wq * 1e3, z, "tab:orange", ls="--", lw=2, label="3 Hermite elements")
a3.set_xlabel("w (mm)"); a3.legend(fontsize=8); a3.set_title("same bow: nodes exact, cubic per zone", fontsize=10)
print(f"nodal loads (u, phi) per node: {np.round(f.reshape(-1, 2), 1).tolist()} (N, N m); "
      f"max |w_FE - w_cf| = {np.abs(wq - w_cf).max():.1e} m")
plt.tight_layout()

In [ ]:
# FEM under the VP1 hypotheses (§4.2). Needs the VP1 closed-form cell (§4.1) (c, E, alpha, I, A, D, dT, dT_m, L_c, r, R1, R2);
# earlier cells (§2.1-§2.2, §3.1, the §4.1 figures, the cell above) rebind H, EI, zb, zt, k0, k, x, K, f, so they are set again here
zb, zt, H = c.stations["core_bottom"], c.stations["core_top"], c.H
k0 = alpha * dT / D
EI, EA = E * I, E * A
k = lambda z: k0 * r(z)                      # free curvature of the y plane (1/m)

# bending on the minimal mesh: 3 elements, kinks on nodes
x = np.array([0.0, zb, zt, H])
zg, xg, wg = solver.gauss_points(x, 3)
K, f = solver.assemble(x, EI, k(zg), xg, wg)
for i in np.flatnonzero(np.abs(f) > 1e-9 * EI * k0):
    print(f"assembled thermal load: {'force' if i % 2 == 0 else 'moment'} {f[i]:+.3f} {'N' if i % 2 == 0 else 'N m'} at z = {x[i // 2]} m")
print(f"expected: +EI k0/L_c = {EI * k0 / L_c:.3f} N at {zb} m, -EI k0/L_c at {zt} m, EI k0 = {EI * k0:.3f} N m at {H} m")
U, R = solver.solve(K, f, [0, 1])
zz = np.linspace(0, H, 401)
u, du, d2u = solver.evaluate(x, U, zz)
print(f"minimal mesh: max |w - k0 R2| = {np.abs(u - k0 * R2(zz)).max():.1e} m, "
      f"max |theta - k0 R1| = {np.abs(du - k0 * R1(zz)).max():.1e} rad, "
      f"max |m_y| = {np.abs(EI * (d2u - k(zz))).max():.1e} N m, reactions {R}")

# axial rod on the case mesh (nz elements + stations)
x = solver.mesh(H, int(c.geo["nz"]), list(c.stations.values()))
zg, xg, wg = solver.gauss_points(x, 3)
K, f = solver.assemble_rod(x, EA, alpha * dT_m * r(zg), xg, wg)
Uz, _ = solver.solve(K, f, [0])
zm = 0.5 * (x[:-1] + x[1:])
Nm = EA * (solver.evaluate_rod(x, Uz, zm)[1] - alpha * dT_m * r(zm))
_, duz_end = solver.evaluate_rod(x, Uz, x[:-1] + 1e-9)
h = np.diff(x).max()
print(f"rod, L_e = {h:.3g} m: nodes max |u_z - alpha dT_m R1| = {np.abs(Uz - alpha * dT_m * R1(x)).max():.1e} m, "
      f"element midpoints {np.abs(solver.evaluate_rod(x, Uz, zm)[0] - alpha * dT_m * R1(zm)).max() * 1e6:.3f} um "
      f"(bound L_e^2 alpha dT_m / (8 L_c) = {h**2 * alpha * dT_m / (8 * L_c) * 1e6:.2f} um)")
print(f"rod N: midpoints max |N| = {np.abs(Nm).max():.1e} N, element ends max |N| = "
      f"{np.abs(EA * (duz_end - alpha * dT_m * r(x[:-1]))).max() / 1e3:.1f} kN "
      f"(EA alpha dT_m L_e / (2 L_c) = {EA * alpha * dT_m * h / (2 * L_c) / 1e3:.1f} kN)")

### 4.3 Comparison with the IAEA document

IWGFR/75 Part I compares Example 1 with two references:
- the **linear theory**: DT1BOW for the bow (PDF p. 52: 1.000 / 3.250 / 12.250 mm at core top, LRP, URP) and DT1LEN for the axial expansion from 20 °C (PDF p. 51: 18.225 / 22.770 / 31.860 mm);
- the **AVERAGE(10)** of the participants at the URP, with its spread (AT1BOW, paras 17–19): spread = (max − min)/average. RAINBOW (Mitsubishi) is left out, 1.8 % below the average of all 11.

The participants' values are those of `U_hex_shell/bowing/corner-corner/IAEA/iwgfr.py` (DT1BOW; IWGFR/76 Part II where DT1BOW is not legible). The average and spread at core top and LRP are computed here from the same 10 codes; the IAEA declares them only at the URP.

**Linear theory.** IWGFR/75 derives it in Part I, App. 5 (pp. 199–202). In our notation:
- the curvature is $k = k_0\,\eta(z)$, with $k_0 = \alpha\Delta T/D_m$ (App. 5, Eq. 1);
- the deflection is $u_y = k_0 (z - z_b)^3/(6L_c)$ in the core (Eq. 8);
- above the core it is $u_y = k_0\left[3(z - z_b)^2 - 3L_c(z - z_b) + L_c^2\right]/6$ (Eq. 9), which equals $k_0R_2$ written from $z_b$.

This is the closed form of §4.1, so w and $u_z$ (from 20 °C) match DT1BOW and DT1LEN to the printed digits.

**Participants.** At the URP the AVERAGE(10) is 12.259 mm, 0.07 % above the closed form, with a spread of 1.24 % as declared (1.25 % from the range 12.180–12.333 mm). The closed form lies inside the band at every station.

**Axial expansion.** Six codes give 31.86–31.97 mm at the URP from 20 °C (AT1LEN; AVERAGE(6) 31.90 mm, spread 0.35 %), against 31.860 mm of the closed form. Belgium (STRAW) gives 33.30 mm, measured from 0 °C.

**Second-order effects** (Part I, §16, p. 13, and App. 5; DT1BOW, URP): 12.47 mm with axial elongation, 12.14 mm with lateral dilation, 12.36 mm with both. They are outside the hypotheses of §4.1 and are listed for scale only.


In [ ]:
# §4.3, closed form (§4.1) against the IAEA document: linear theory and participants (IWGFR/75 DT1BOW, AT1BOW, AT1LEN).
# Needs the VP1 closed-form cell (§4.1) (c, alpha, dT, D, dT_m, T_in, R1, R2)
sys.path.insert(0, os.path.normpath(os.path.join(BASE, "..", "U_hex_shell", "bowing", "corner-corner", "IAEA")))
import iwgfr

k0 = alpha * dT / D
z_st = np.array([c.stations["core_top"], c.stations["LRP"], c.H])
w_cf = k0 * R2(z_st) * 1e3                                       # (mm)
uz20_cf = alpha * ((T_in - 293.15) * z_st + dT_m * R1(z_st)) * 1e3   # (mm), lengths from 20 °C
bd = iwgfr.band("ex1")
rows = []
for j, st in enumerate(iwgfr.STATIONS):
    b = bd[st]
    rows.append([st, f"{w_cf[j]:.4f}", f"{iwgfr.THEORY['ex1'][j]:.3f}", (f"{b['average']:.3f} (IAEA)" if b["declared"] else f"{b['average']:.4f}"),
                 f"{b['spread']:.2f}" + (" (IAEA)" if b["declared"] else ""), f"{b['min']:.3f}–{b['max']:.3f}", f"{100 * (w_cf[j] - b['average']) / b['average']:+.2f}",
                 f"{uz20_cf[j]:.3f}", f"{iwgfr.AXIAL_THEORY[j]:.3f}"])
table(rows, ["station", "closed form w (mm)", "DT1BOW theory (mm)", "AVERAGE(10) (mm)", "spread (%)", "range (mm)",
             "closed form − average (%)", "closed form u_z, 20 °C (mm)", "DT1LEN theory (mm)"])
avg6, spr6 = iwgfr.AXIAL_DECLARED["AVERAGE(6)"]
print(f"axial, URP: closed form {uz20_cf[-1]:.3f} mm, AT1LEN AVERAGE(6) {avg6:.2f} mm (spread {spr6:.2f} %), "
      f"codes {min(iwgfr.AXIAL_URP.values()):.2f}-{max(iwgfr.AXIAL_URP.values()):.2f} mm")
print("second order, URP (mm):", iwgfr.SECOND_ORDER_URP["ex1"])

fig, axs = plt.subplots(1, 3, figsize=(15, 4.8))
for j, (a, st) in enumerate(zip(axs, iwgfr.STATIONS)):
    names = list(iwgfr.CODES)
    yv = [iwgfr.CODES[n]["ex1"][j] for n in names]
    cols = ["0.6" if n == iwgfr.REJECTED else "tab:blue" for n in names]
    a.scatter(yv, range(len(names)), c=cols, zorder=3)
    b = bd[st]
    a.axvspan(b["min"], b["max"], color="tab:blue", alpha=0.1, label="range of AVERAGE(10)")
    a.axvline(b["average"], color="tab:blue", ls="--", lw=1, label="AVERAGE(10)")
    a.axvline(w_cf[j], color="k", lw=2, label="closed form (§4.1)")
    a.set_yticks(range(len(names)), names if j == 0 else [""] * len(names), fontsize=8)
    a.set_xlabel("w (mm)"); a.set_title(st, fontsize=10)
axs[0].legend(fontsize=8, loc="lower left")
fig.suptitle("IAEA VP1, example 1: closed form vs the participants of IWGFR/75 (grey: RAINBOW, out of AVERAGE(10))", fontsize=10)
plt.tight_layout()


### 4.4 Comparison with Z3ST 3D

**The Z3ST 3D run (the reference).** This is `U_hex_shell/bowing/corner-corner/IAEA/VP1`:
- `regime: 3d`, vector Lagrange of order `mechanical.order` (default 1; not set in that input);
- the wall mesh with nf = 8 elements per flat, nt = 1 through the thickness and nz = 400 along z;
- Hooke isotropic, $\boldsymbol\sigma = \lambda\,\mathrm{tr}(\boldsymbol\varepsilon - \varepsilon^*\mathbf 1)\,\mathbf 1 + 2\mu\,(\boldsymbol\varepsilon - \varepsilon^*\mathbf 1)$, with $\boldsymbol\varepsilon = \mathrm{sym}\,\nabla\mathbf u$ and $\varepsilon^* = \alpha(T - T_\mathrm{ref})$, the same E, α and $T_\mathrm{ref}$;
- input: the VP1 T string as Dirichlet on the faces, base clamped.

It shares small strains, constant E and α and the undeformed geometry, but **not** Euler–Bernoulli: sections are free to warp, shear and distort, and ν acts in the plane of the section. The gap to §4.1, −0.15 % at the URP (below), therefore contains what the beam leaves out (the thermal shear) plus the Z3ST discretisation error. The −0.04 % rest is still open.

**Z3ST `regime: 1d` (native).** A line mesh with a uniaxial stress state, $\sigma = E\,(\varepsilon - \varepsilon^*)$ (`models/mechanical_model.py`), on a line laid along x (`epsilon()` takes $u_{,x}$).
- With the section-mean T as input it is expected to solve the rod of §4.2 point 2 and give back $u_z$ of §4.1 step 5. This is not yet checked by a run.
- It has no rotation dof and no section, hence no $\kappa^*_a$ and no bow.


**The Z3ST VP1 case, as a beam:**
- the same `D` ($D_o$), `t`, `H`, `orientation`;
- the same material card (`15_15Ti_const.yaml`);
- the same T string (imposed on every face in the 3D run, integrated over the wall here).

**Closed form** (§1.6, §4.1). T is linear across the section at every height, from the cold corner (+y) to the hot one. Over the core ($z_b = 1.5$ to $z_t = 2.5$ m) the gradient ramps up linearly to $k_0 = \alpha\,\Delta T/D_m = 1.8\times10^{-5}\cdot 50/0.150 = 6.0\times10^{-3}$ 1/m, and stays constant above. This gives w = 1.000, 3.250 and 12.250 mm at the core top, LRP and URP (the IWGFR linear theory). For the axial part, the mean T rise is 125 K above the core, so $u_z = \alpha\,125\,\mathrm{K}\int_0^z \eta\,d\zeta = 1.125, 2.250, 4.500$ mm.

The FE nodes are exact (§2.1). VP1 runs with nz = 80 (h = 0.05 m, every station a node): the condition number of K is 2·10⁸ and the round-off on w is 5·10⁻¹⁰ relative at the URP, 3·10⁻⁹ at the core top, within the 1e-8 non-regression tolerance. With nz = 400 it grows to 10¹¹ and 2·10⁻⁶ (§2.4), and the tolerance would not hold. The comparison with Z3ST is in `VP1/vp1_vs_z3st.py`. At the URP, Z3ST − beam = −18.9 µm (−0.15 %): −13.8 µm is the thermal shear of the core ($\gamma = -c\,k'$, $c = 0.410\,a_m^2$), the rest is a −0.04 % curvature deficit above the core (to be checked on `fine_hex`). The axial displacement u_z agrees to 0.1 µm.

In [ ]:
c, out = run(VP1)
zs = out["z_s"]
zb, zt, k0, L_c = 1.5, 2.5, 1.8e-5 * 25 / 0.075, 1.0
w_cf = np.where(zs <= zb, 0, np.where(zs <= zt, k0 * (zs - zb)**3 / (6 * L_c), k0 * (L_c**2 / 6 + L_c * (zs - zt) / 2 + (zs - zt)**2 / 2)))
uz_cf = 1.8e-5 * 125 * np.where(zs <= zb, 0, np.where(zs <= zt, (zs - zb)**2 / (2 * L_c), L_c / 2 + zs - zt))
fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(15, 4.8), sharey=True)
a1.plot(-out["ky_s"] * 1e3, zs, label=r"$\kappa^*$ towards the cold corner"); a1.set_xlabel("1/km")
a2.plot(out["uy_s"] * 1e3, zs, lw=2, label="beam w = u_y"); a2.plot(w_cf * 1e3, zs, "k--", label="closed form"); a2.set_xlabel("w (mm)")
a3.plot(out["uz_s"] * 1e3, zs, lw=2, color="C2", label="beam u_z"); a3.plot(uz_cf * 1e3, zs, "k--", label="closed form"); a3.set_xlabel("u_z (mm)")
for a in (a1, a2, a3):
    a.axhline(zb, color="grey", ls=":"); a.axhline(zt, color="grey", ls=":"); a.legend(fontsize=9)
a1.set_ylabel("z (m)")
plt.tight_layout()
w_at = lambda zz: float(solver.evaluate(out["z"], out["U_y"], zz)[0][0])
print("w (core top, LRP, URP) =", [round(w_at(z) * 1e3, 6) for z in (2.5, 3.0, 4.0)], "mm (IWGFR: 1.000, 3.250, 12.250)")
print(f"u_z(URP) = {out['U_z'][-1] * 1e3:.6f} mm, max |sigma_zz| = {np.abs(out['sigma_mid']).max():.2e} Pa (T linear in the section)")

In [ ]:
nr = json.load(open(os.path.join(VP1, "output", "non-regression.json")))
res = nr.get("results", nr)
print(f"VP1 non-regression: {nr.get('summary')}, {len(res)} checks, max rel. error "
      f"{max(v['rel_error'] for v in res.values()):.2e}")
summ = json.load(open(os.path.join(VP1, "output", "IAEA_VP1_beam1d_vs_z3st_summary.json")))
table([[r["station"], f"{r['theory_mm']:.3f}", f"{r['beam_mm']:.4f}", f"{r['z3st_mm']:.4f}",
        f"{r['z3st_minus_beam_mm'] * 1e3:+.1f}", f"{r['thermal_shear_mm'] * 1e3:+.1f}", f"{r['rest_mm'] * 1e3:+.1f}",
        f"{r['uz_beam_mm']:.4f}", f"{r['uz_z3st_mm']:.4f}"] for r in summ["rows"]],
      ["station", "IWGFR (mm)", "beam w", "Z3ST w", "Z3ST − beam (µm)", "thermal shear (µm)", "rest (µm)",
       "beam u_z (mm)", "Z3ST u_z (mm)"])
display(Image(os.path.join(VP1, "output", "IAEA_VP1_beam1d_vs_z3st.png"), width=950))